# Beating 81%: Optimizing the Follow Last Result Strategy

We know "Follow Last Result" gives 81% win rate on EURUSD-OTC. Can we do better?

**Baseline:** 81% win rate, 0 martingale busts, ~$6,264 profit on 10K trades

**Ideas to explore:**
1. **Skip uncertain moments** — are there conditions where the 81% drops? Skip those.
2. **Different entry times** — is :25 or :35 better than :30?
3. **Streak-aware betting** — after N consecutive same-direction results, does momentum weaken?
4. **Volatility filter** — does the strategy work better in calm vs volatile periods?
5. **Multi-minute lookahead** — does the pattern extend beyond 1 minute?
6. **Different OTC pairs** — does this work on GBPUSD-OTC, USDJPY-OTC?
7. **Optimal martingale depth** — is 8 levels the sweet spot?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, random, warnings
warnings.filterwarnings('ignore')
random.seed(42)

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)

# Load data
data_files = ['../data/eurusd_otc_all.csv', '../data/eurusd_otc_live.csv', '../data/eurusd_otc_5s.csv']
dfs = [pd.read_csv(f) for f in data_files if os.path.exists(f)]
df = pd.concat(dfs, ignore_index=True).drop_duplicates(subset='timestamp').sort_values('timestamp').reset_index(drop=True)
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
df = df.set_index('datetime').sort_index()
df['minute'] = df.index.minute
df['second'] = df.index.second
df['hour'] = df.index.hour
df['return'] = df['close'].pct_change()

# Build trade data
candles_30 = df[df['second'] == 30][['close', 'minute', 'hour']].copy()
candles_30['future_close'] = candles_30['close'].shift(-6)
candles_30['went_up'] = (candles_30['future_close'] > candles_30['close']).astype(int)
candles_30['prev_went_up'] = candles_30['went_up'].shift(1)
candles_30['matched'] = (candles_30['went_up'] == candles_30['prev_went_up']).astype(int)
candles_30['price_change'] = (candles_30['future_close'] - candles_30['close']).abs()
candles_30 = candles_30.dropna()

baseline = candles_30['matched'].mean()
print(f"Baseline: Follow Last Result = {baseline:.1%}")
print(f"Trades: {len(candles_30):,}")
print(f"\nLet's see if we can beat {baseline:.1%}...")

## 1. When Does 81% Drop? (Skip Conditions)

If we can identify when momentum weakens, we can skip those trades and boost our win rate.

In [ ]:
# How many consecutive same-direction results before momentum breaks?
# After 1 match, 2 matches, 3 matches... does the next one still match?

consec_matches = []
count = 0
for val in candles_30['matched'].values:
    if val == 1:
        count += 1
    else:
        consec_matches.append(count)
        count = 0
if count > 0:
    consec_matches.append(count)

# After N consecutive correct "follow" predictions, what's the probability the next one is also correct?
candles_30['consec_correct'] = 0
count = 0
consec_col = []
for val in candles_30['matched'].values:
    consec_col.append(count)
    if val == 1:
        count += 1
    else:
        count = 0
candles_30['consec_correct'] = consec_col

by_streak = candles_30.groupby('consec_correct').agg(
    win_rate=('matched', 'mean'),
    count=('matched', 'count'),
).reset_index()
by_streak = by_streak[by_streak['count'] >= 20]  # Need enough samples

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['lime' if w > baseline else 'red' for w in by_streak['win_rate']]
axes[0].bar(by_streak['consec_correct'], by_streak['win_rate'], color=colors, alpha=0.7)
axes[0].axhline(y=baseline, color='yellow', linestyle='--', label=f'Baseline ({baseline:.1%})')
axes[0].set_title('Win Rate After N Consecutive Correct Predictions')
axes[0].set_xlabel('Consecutive correct predictions so far')
axes[0].set_ylabel('P(next prediction correct)')
axes[0].legend()

axes[1].bar(by_streak['consec_correct'], by_streak['count'], color='cyan', alpha=0.5)
axes[1].set_title('Sample Count')
axes[1].set_xlabel('Consecutive correct predictions')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print("After N consecutive correct Follow predictions:")
for _, row in by_streak.iterrows():
    better = "BETTER" if row['win_rate'] > baseline else "worse"
    print(f"  After {int(row['consec_correct']):2d} correct: {row['win_rate']:.1%} ({int(row['count'])} samples) — {better}")

## 2. Optimal Entry Time

We trade at :30. Would :25 or :35 be better? The later you enter, the more you know — but the shorter the prediction window.

In [ ]:
# Test different entry seconds
entry_results = []

for entry_second in [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55]:
    # How many 5s candles until next :00?
    candles_ahead = (60 - entry_second) // 5
    
    candles_entry = df[df['second'] == entry_second][['close']].copy()
    candles_entry['future_close'] = candles_entry['close'].shift(-candles_ahead)
    candles_entry['went_up'] = (candles_entry['future_close'] > candles_entry['close']).astype(int)
    candles_entry['prev_went_up'] = candles_entry['went_up'].shift(1)
    candles_entry = candles_entry.dropna()
    
    match_rate = (candles_entry['went_up'] == candles_entry['prev_went_up']).mean()
    
    entry_results.append({
        'entry_second': entry_second,
        'prediction_window': 60 - entry_second,
        'match_rate': match_rate,
        'count': len(candles_entry),
    })

entry_df = pd.DataFrame(entry_results)

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['lime' if m > baseline else ('gold' if m > 0.5405 else 'red') for m in entry_df['match_rate']]
bars = ax.bar(entry_df['entry_second'].astype(str), entry_df['match_rate'], color=colors, alpha=0.8)
ax.axhline(y=baseline, color='yellow', linestyle='--', label=f'Current baseline ({baseline:.1%})')
ax.axhline(y=0.5405, color='red', linestyle=':', label='Break-even (54%)')
ax.set_title('Follow Last Result Win Rate by Entry Second')
ax.set_xlabel('Entry second within minute')
ax.set_ylabel('Win Rate')
ax.legend()
plt.tight_layout()
plt.show()

print(f"{'Entry':>6} {'Window':>8} {'WinRate':>8} {'Samples':>8} {'vs Baseline':>12}")
print("-" * 50)
for _, row in entry_df.iterrows():
    diff = row['match_rate'] - baseline
    marker = " ★ BEST" if row['match_rate'] == entry_df['match_rate'].max() else ""
    print(f"  :{int(row['entry_second']):02d}   {int(row['prediction_window']):>5}s   {row['match_rate']:>7.1%}   {int(row['count']):>7}   {diff:>+.1%}{marker}")

## 3. Volatility Filter

Does the strategy work better when price is moving a lot (volatile) vs barely moving (quiet)?

In [ ]:
# Measure volatility as the absolute price change of the PREVIOUS trade
candles_30['prev_change'] = candles_30['price_change'].shift(1)
candles_30['prev_change_pct'] = (candles_30['prev_change'] / candles_30['close']) * 100

# Bin by previous move size
candles_30['volatility_bin'] = pd.qcut(
    candles_30['prev_change_pct'].dropna(), q=5, 
    labels=['Very quiet', 'Quiet', 'Normal', 'Active', 'Very active'],
    duplicates='drop'
)

by_vol = candles_30.groupby('volatility_bin').agg(
    win_rate=('matched', 'mean'),
    count=('matched', 'count'),
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['lime' if w > baseline else 'red' for w in by_vol['win_rate']]
ax.bar(by_vol['volatility_bin'].astype(str), by_vol['win_rate'], color=colors, alpha=0.7)
ax.axhline(y=baseline, color='yellow', linestyle='--', label=f'Baseline ({baseline:.1%})')
ax.set_title('Follow Last Result Win Rate by Previous Move Size')
ax.set_ylabel('Win Rate')
ax.set_xlabel('Previous trade move size')
ax.legend()
ax.set_ylim(0.7, 0.9)
plt.tight_layout()
plt.show()

for _, row in by_vol.iterrows():
    diff = row['win_rate'] - baseline
    print(f"  {row['volatility_bin']:>12}: {row['win_rate']:.1%} ({int(row['count'])} trades) {diff:>+.1%}")

## 4. Multi-Minute Momentum

Does the pattern extend? If we skip a minute and look 2 or 3 minutes ahead, is there still momentum?

In [ ]:
# Does minute N's direction predict minute N+2, N+3, etc.?
multi_results = []
for lag in range(1, 11):
    lagged = candles_30['went_up'].shift(lag)
    valid = candles_30.dropna(subset=['went_up'])
    valid_lag = lagged.dropna()
    common = valid.index.intersection(valid_lag.index)
    
    current = candles_30.loc[common, 'went_up']
    prev = lagged.loc[common]
    match = (current == prev).mean()
    
    multi_results.append({'lag': lag, 'match_rate': match, 'minutes_back': lag})

multi_df = pd.DataFrame(multi_results)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['lime' if m > 0.5405 else 'red' for m in multi_df['match_rate']]
ax.bar(multi_df['lag'], multi_df['match_rate'], color=colors, alpha=0.7)
ax.axhline(y=baseline, color='yellow', linestyle='--', label=f'Lag 1 ({baseline:.1%})')
ax.axhline(y=0.5405, color='red', linestyle=':', label='Break-even')
ax.axhline(y=0.5, color='white', linestyle=':', alpha=0.3)
ax.set_title('Follow Result from N Minutes Ago')
ax.set_xlabel('Lag (minutes back)')
ax.set_ylabel('Win Rate')
ax.legend()
plt.tight_layout()
plt.show()

for _, row in multi_df.iterrows():
    status = "profitable" if row['match_rate'] > 0.5405 else "NOT profitable"
    print(f"  Lag {int(row['lag'])}: {row['match_rate']:.1%} — {status}")

## 5. Combined Filters

What if we combine the best conditions? Only trade when:
- We're on a winning streak (momentum strong)
- Previous move was large enough (high volatility)
- Best entry time

In [ ]:
# Test combinations of filters
combos = []

# No filter (baseline)
mask_all = pd.Series(True, index=candles_30.index)
wr = candles_30.loc[mask_all, 'matched'].mean()
combos.append({'name': 'Baseline (no filter)', 'win_rate': wr, 'trades': mask_all.sum()})

# Only after 1+ consecutive correct
for min_streak in [1, 2, 3, 5]:
    mask = candles_30['consec_correct'] >= min_streak
    if mask.sum() > 50:
        wr = candles_30.loc[mask, 'matched'].mean()
        combos.append({'name': f'After {min_streak}+ correct streak', 'win_rate': wr, 'trades': mask.sum()})

# Only high volatility
for vol_label in ['Active', 'Very active']:
    mask = candles_30['volatility_bin'] == vol_label
    if mask.sum() > 50:
        wr = candles_30.loc[mask, 'matched'].mean()
        combos.append({'name': f'Volatility: {vol_label}', 'win_rate': wr, 'trades': mask.sum()})

# Only low volatility
for vol_label in ['Very quiet', 'Quiet']:
    mask = candles_30['volatility_bin'] == vol_label
    if mask.sum() > 50:
        wr = candles_30.loc[mask, 'matched'].mean()
        combos.append({'name': f'Volatility: {vol_label}', 'win_rate': wr, 'trades': mask.sum()})

# Best hours (if any stood out)
for hour_range, label in [((0, 6), 'Night 0-6'), ((6, 12), 'Morning 6-12'), 
                           ((12, 18), 'Afternoon 12-18'), ((18, 24), 'Evening 18-24')]:
    mask = (candles_30['hour'] >= hour_range[0]) & (candles_30['hour'] < hour_range[1])
    if mask.sum() > 50:
        wr = candles_30.loc[mask, 'matched'].mean()
        combos.append({'name': f'Hours: {label}', 'win_rate': wr, 'trades': mask.sum()})

# Streak + volatility combos
for min_streak in [1, 2]:
    for vol_label in ['Active', 'Very active']:
        mask = (candles_30['consec_correct'] >= min_streak) & (candles_30['volatility_bin'] == vol_label)
        if mask.sum() > 30:
            wr = candles_30.loc[mask, 'matched'].mean()
            combos.append({'name': f'Streak {min_streak}+ AND {vol_label}', 'win_rate': wr, 'trades': mask.sum()})

combo_df = pd.DataFrame(combos).sort_values('win_rate', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
colors = ['lime' if w > baseline else ('gold' if w > 0.5405 else 'red') for w in combo_df['win_rate']]
bars = ax.barh(range(len(combo_df)), combo_df['win_rate'], color=colors, alpha=0.7)
ax.set_yticks(range(len(combo_df)))
ax.set_yticklabels(combo_df['name'], fontsize=8)
ax.axvline(x=baseline, color='yellow', linestyle='--', label=f'Baseline ({baseline:.1%})')
ax.set_title('Filter Combinations — Can We Beat 81%?')
ax.set_xlabel('Win Rate')
ax.legend()

# Add trade count labels
for i, (_, row) in enumerate(combo_df.iterrows()):
    ax.text(row['win_rate'] + 0.002, i, f"n={int(row['trades'])}", va='center', fontsize=7)

plt.tight_layout()
plt.show()

print(f"\n{'Filter':<40} {'WinRate':>8} {'Trades':>8} {'vs Base':>8}")
print("-" * 68)
for _, row in combo_df.iterrows():
    diff = row['win_rate'] - baseline
    marker = " ★" if row['win_rate'] > baseline else ""
    print(f"{row['name']:<40} {row['win_rate']:>7.1%} {int(row['trades']):>8} {diff:>+7.1%}{marker}")

## 6. Full Backtest: Simulate Exact Bot Behavior

Run both v1 (:30 entry) and v2 (:00 entry + volatility filters) against all 122K candles. Simulate martingale, track P&L, count busts.

In [ ]:
def full_backtest(df, entry_second, min_move=0.0, base_stake=1.0, payout=0.85, max_losses=8, max_exposure=200):
    """
    Simulate exact bot behavior:
    1. At entry_second of each minute, check if we should trade
    2. Follow Last Result direction
    3. Martingale stake progression
    4. Optional volatility filter (min_move)
    """
    # Build trade-level data at the given entry second
    candles_entry = df[df['second'] == entry_second][['close']].copy()
    candles_ahead = (60 - entry_second) // 5
    candles_entry['future_close'] = candles_entry['close'].shift(-candles_ahead)
    candles_entry['went_up'] = (candles_entry['future_close'] > candles_entry['close']).astype(int)
    candles_entry['prev_went_up'] = candles_entry['went_up'].shift(1)
    candles_entry['prev_move'] = (candles_entry['future_close'].shift(1) - candles_entry['close'].shift(1)).abs()
    candles_entry = candles_entry.dropna()
    
    # Simulate
    balance = 1000
    history = [balance]
    stake = base_stake
    consec = 0
    wins = 0
    losses = 0
    busts = 0
    skips = 0
    last_result_dir = None
    
    for _, row in candles_entry.iterrows():
        # Volatility filter
        if min_move > 0 and row['prev_move'] < min_move:
            skips += 1
            continue
        
        # Direction: follow last result
        if last_result_dir is None:
            our_bet = random.randint(0, 1)  # First trade random
        else:
            our_bet = last_result_dir
        
        actual = int(row['went_up'])
        
        # Martingale safety
        if consec >= max_losses or stake > max_exposure:
            busts += 1
            stake = base_stake
            consec = 0
        
        if stake > balance:
            break
        
        # Result
        if our_bet == actual:
            balance += stake * payout
            wins += 1
            last_result_dir = actual
            stake = base_stake
            consec = 0
        else:
            balance -= stake
            losses += 1
            consec += 1
            last_result_dir = actual  # Follow what ACTUALLY happened
            stake = round((stake + base_stake) / payout, 2)
        
        history.append(balance)
    
    total = wins + losses
    return {
        'balance': balance,
        'profit': balance - 1000,
        'wins': wins,
        'losses': losses,
        'skips': skips,
        'trades': total,
        'win_rate': wins / total if total > 0 else 0,
        'busts': busts,
        'max_dd': 1000 - min(history),
        'history': history,
    }


# Run all configurations
configs = [
    {'label': 'v1: Entry :30, no filter', 'entry': 30, 'min_move': 0.0},
    {'label': 'v2: Entry :00, no filter', 'entry': 0, 'min_move': 0.0},
    {'label': 'v2: Entry :00, Active filter', 'entry': 0, 'min_move': 0.000020},
    {'label': 'v2: Entry :00, Very Active filter', 'entry': 0, 'min_move': 0.000040},
    {'label': 'v2: Entry :05, no filter', 'entry': 5, 'min_move': 0.0},
    {'label': 'v2: Entry :10, no filter', 'entry': 10, 'min_move': 0.0},
    {'label': 'v2: Entry :00, Active + 8 martingale', 'entry': 0, 'min_move': 0.000020},
]

print(f"FULL BACKTEST — Follow Last Result on 122K candles")
print(f"Starting balance: $1,000 | Base stake: $1 | Payout: 85% | Max losses: 8")
print(f"{'='*95}")
print(f"{'Config':<40} {'WinRate':>8} {'Trades':>7} {'Skips':>7} {'Busts':>6} {'Profit':>10} {'MaxDD':>8}")
print(f"{'-'*95}")

all_results = []
for cfg in configs:
    r = full_backtest(df, entry_second=cfg['entry'], min_move=cfg['min_move'])
    r['label'] = cfg['label']
    all_results.append(r)
    marker = " ★" if r['profit'] > 0 else ""
    print(f"{cfg['label']:<40} {r['win_rate']:>7.1%} {r['trades']:>7} {r['skips']:>7} "
          f"{r['busts']:>6} ${r['profit']:>9.2f} ${r['max_dd']:>7.2f}{marker}")

# Plot equity curves
fig, ax = plt.subplots(figsize=(16, 7))
for r in all_results:
    lw = 2 if 'Active' in r['label'] else 1
    ax.plot(r['history'], label=f"{r['label']} (${r['profit']:.0f})", linewidth=lw)
ax.axhline(y=1000, color='yellow', linestyle='--', alpha=0.5, label='Starting balance')
ax.set_title('Full Backtest: Follow Last Result — All Configurations')
ax.set_xlabel('Trade #')
ax.set_ylabel('Balance ($)')
ax.legend(fontsize=7, loc='upper left')
plt.tight_layout()
plt.show()

# Best config
best = max(all_results, key=lambda x: x['profit'])
print(f"\nBest: {best['label']}")
print(f"  Win rate: {best['win_rate']:.1%}")
print(f"  Profit:   ${best['profit']:.2f}")
print(f"  Trades:   {best['trades']}")
print(f"  Busts:    {best['busts']}")
print(f"  Per trade: ${best['profit']/best['trades']:.3f}") if best['trades'] > 0 else None
print(f"  Per hour:  ${best['profit'] / (len(df) * 5 / 3600):.2f}")

## 7. Breaking 90%: Advanced Optimizations

Six ideas to push past 87%.

In [ ]:
# Rebuild data at :00 entry for all optimizations
entry_sec = 0
candles_ahead = 12  # :00 to next :00 = 60s = 12 five-second candles

c00 = df[df['second'] == entry_sec][['close']].copy()
c00['future_close'] = c00['close'].shift(-candles_ahead)
c00['went_up'] = (c00['future_close'] > c00['close']).astype(int)
c00['prev_went_up_1'] = c00['went_up'].shift(1)  # Lag 1
c00['prev_went_up_2'] = c00['went_up'].shift(2)  # Lag 2
c00['prev_went_up_3'] = c00['went_up'].shift(3)  # Lag 3
c00['prev_move'] = (c00['future_close'].shift(1) - c00['close'].shift(1)).abs()

# Price at :55 of previous minute (5 seconds before our entry)
c55 = df[df['second'] == 55][['close']].rename(columns={'close': 'price_55'})
c00['price_55'] = None
for idx in c00.index:
    target = idx - pd.Timedelta(seconds=5)
    matches = c55.index.get_indexer([target], method='nearest')
    if matches[0] >= 0:
        c00.loc[idx, 'price_55'] = c55.iloc[matches[0]]['price_55']

c00['last_5s_up'] = (c00['close'] > c00['price_55'].astype(float)).astype(int)
c00 = c00.dropna()

baseline_wr = (c00['went_up'] == c00['prev_went_up_1']).mean()
print(f"Baseline (follow lag 1 at :00): {baseline_wr:.1%}")
print(f"Samples: {len(c00):,}\n")

# ═══════════════════════════════════════════════════════════════
# IDEA 1: Follow first 30s of current minute
# At :00 we can see what just happened from :30 to :00
# ═══════════════════════════════════════════════════════════════
print("IDEA 1: Follow the move from :30 to :00 (what we just witnessed)")
print("-" * 60)

# The last_5s_up tells us if price went up from :55 to :00
# But we really want :30 to :00
c30 = df[df['second'] == 30][['close']].rename(columns={'close': 'price_30'})
c00['price_30'] = None
for idx in c00.index:
    target = idx - pd.Timedelta(seconds=30)
    matches = c30.index.get_indexer([target], method='nearest')
    if matches[0] >= 0:
        c00.loc[idx, 'price_30'] = c30.iloc[matches[0]]['price_30']

c00['last_30s_up'] = (c00['close'] > c00['price_30'].astype(float)).astype(int)

# Follow what we just saw vs follow last trade result
follow_30s = (c00['went_up'] == c00['last_30s_up']).mean()
follow_lag1 = (c00['went_up'] == c00['prev_went_up_1']).mean()
print(f"  Follow last 30s move:   {follow_30s:.1%}")
print(f"  Follow last result:     {follow_lag1:.1%}")
print(f"  Better: {'30s move' if follow_30s > follow_lag1 else 'Last result'}")

# ═══════════════════════════════════════════════════════════════
# IDEA 2: Size-weighted confidence
# ═══════════════════════════════════════════════════════════════
print(f"\nIDEA 2: Win rate by size of last move")
print("-" * 60)

c00['prev_move_q'] = pd.qcut(c00['prev_move'], q=10, duplicates='drop')
by_size = c00.groupby('prev_move_q').apply(
    lambda g: (g['went_up'] == g['prev_went_up_1']).mean()
).reset_index()
by_size.columns = ['size_bin', 'win_rate']
for _, row in by_size.iterrows():
    marker = " ★" if row['win_rate'] > baseline_wr else ""
    print(f"  {str(row['size_bin']):>30}: {row['win_rate']:.1%}{marker}")

# ═══════════════════════════════════════════════════════════════
# IDEA 3: Combine lag 1 + lag 2
# ═══════════════════════════════════════════════════════════════
print(f"\nIDEA 3: Combine lag 1 AND lag 2")
print("-" * 60)

both_agree = c00['prev_went_up_1'] == c00['prev_went_up_2']
both_disagree = c00['prev_went_up_1'] != c00['prev_went_up_2']

if both_agree.sum() > 0:
    wr_agree = (c00.loc[both_agree, 'went_up'] == c00.loc[both_agree, 'prev_went_up_1']).mean()
    print(f"  Lag 1 & Lag 2 AGREE (both same direction): {wr_agree:.1%} ({both_agree.sum()} trades)")

if both_disagree.sum() > 0:
    wr_disagree = (c00.loc[both_disagree, 'went_up'] == c00.loc[both_disagree, 'prev_went_up_1']).mean()
    print(f"  Lag 1 & Lag 2 DISAGREE (different):        {wr_disagree:.1%} ({both_disagree.sum()} trades)")

# Also try: follow lag1, but skip if lag2 disagrees
print(f"\n  Strategy: Follow lag 1, but SKIP when lag 2 disagrees")
print(f"  → Win rate: {wr_agree:.1%} on {both_agree.sum()} trades (skip {both_disagree.sum()})")

# Add lag 3
all_3_agree = both_agree & (c00['prev_went_up_2'] == c00['prev_went_up_3'])
if all_3_agree.sum() > 0:
    wr_3 = (c00.loc[all_3_agree, 'went_up'] == c00.loc[all_3_agree, 'prev_went_up_1']).mean()
    print(f"\n  All 3 lags AGREE: {wr_3:.1%} ({all_3_agree.sum()} trades)")

# ═══════════════════════════════════════════════════════════════
# IDEA 4: Fine-grained entry timing (:00 to :05)
# ═══════════════════════════════════════════════════════════════
print(f"\nIDEA 4: Fine-grained entry timing")
print("-" * 60)

for sec in [0, 1, 2, 3, 4, 5]:
    # We only have 5-second candles, so 0 and 5 are the only exact ones
    # But we can check :00 vs :05
    if sec % 5 == 0:
        ahead = (60 - sec) // 5
        c_sec = df[df['second'] == sec][['close']].copy()
        c_sec['future_close'] = c_sec['close'].shift(-ahead)
        c_sec['went_up'] = (c_sec['future_close'] > c_sec['close']).astype(int)
        c_sec['prev'] = c_sec['went_up'].shift(1)
        c_sec = c_sec.dropna()
        wr = (c_sec['went_up'] == c_sec['prev']).mean()
        marker = " ★ BEST" if sec == 0 else ""
        print(f"  Entry at :{sec:02d} → {wr:.1%} ({len(c_sec)} trades){marker}")

# ═══════════════════════════════════════════════════════════════
# IDEA 5: Skip after a loss
# ═══════════════════════════════════════════════════════════════
print(f"\nIDEA 5: Skip N trades after a loss")
print("-" * 60)

for skip_after_loss in [0, 1, 2, 3]:
    last_dir = None
    skip_count = 0
    correct = 0
    total = 0
    
    for _, row in c00.iterrows():
        if skip_count > 0:
            skip_count -= 1
            last_dir = int(row['went_up'])  # Still track the result
            continue
        
        if last_dir is None:
            last_dir = int(row['went_up'])
            continue
        
        actual = int(row['went_up'])
        if last_dir == actual:
            correct += 1
        else:
            skip_count = skip_after_loss
        
        total += 1
        last_dir = actual
    
    wr = correct / total if total > 0 else 0
    trades = total
    marker = " ★" if wr > baseline_wr else ""
    print(f"  Skip {skip_after_loss} after loss: {wr:.1%} ({trades} trades){marker}")

# ═══════════════════════════════════════════════════════════════
# IDEA 6: Double confirmation (lag 1 + last 30s same direction)
# ═══════════════════════════════════════════════════════════════
print(f"\nIDEA 6: Double confirmation — lag 1 AND last 30s move agree")
print("-" * 60)

double_confirm = c00['prev_went_up_1'] == c00['last_30s_up']
if double_confirm.sum() > 0:
    wr_dc = (c00.loc[double_confirm, 'went_up'] == c00.loc[double_confirm, 'prev_went_up_1']).mean()
    print(f"  Both agree:    {wr_dc:.1%} ({double_confirm.sum()} trades)")
    
double_conflict = c00['prev_went_up_1'] != c00['last_30s_up']
if double_conflict.sum() > 0:
    wr_conflict = (c00.loc[double_conflict, 'went_up'] == c00.loc[double_conflict, 'prev_went_up_1']).mean()
    print(f"  They disagree: {wr_conflict:.1%} ({double_conflict.sum()} trades)")
    print(f"\n  Strategy: Follow lag 1 only when last 30s confirms")
    print(f"  → {wr_dc:.1%} win rate, skip {double_conflict.sum()} conflicting trades")

# ═══════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"SUMMARY — Best ideas to beat {baseline_wr:.1%}")
print(f"{'='*60}")
ideas = [
    (f"Baseline (follow lag 1)", baseline_wr, len(c00)),
    (f"Follow 30s move instead", follow_30s, len(c00)),
    (f"Lag 1+2 agree only", wr_agree, int(both_agree.sum())),
    (f"All 3 lags agree", wr_3, int(all_3_agree.sum())),
    (f"Double confirm (lag1 + 30s)", wr_dc, int(double_confirm.sum())),
]
ideas.sort(key=lambda x: x[1], reverse=True)
for name, wr, trades in ideas:
    diff = wr - baseline_wr
    marker = " ★" if wr > baseline_wr else ""
    print(f"  {name:<35} {wr:>6.1%} ({trades:>5} trades) {diff:>+5.1%}{marker}")

## 8. Combined Strategy: 3 Lags Agree + Skip After Loss

Combine the two best ideas and run a full martingale backtest.

In [ ]:
def combined_backtest(df, entry_second=0, require_lags=1, skip_after_loss=0,
                       base_stake=1.0, payout=0.85, max_losses=8, max_exposure=200):
    """
    Full simulation:
    - Entry at given second
    - Follow last result
    - Only trade when last N lags all agree (require_lags=1,2,3)
    - Skip M minutes after a loss
    - Martingale stake progression
    """
    candles_ahead = (60 - entry_second) // 5
    c = df[df['second'] == entry_second][['close']].copy()
    c['future_close'] = c['close'].shift(-candles_ahead)
    c['went_up'] = (c['future_close'] > c['close']).astype(int)
    
    for lag in range(1, require_lags + 1):
        c[f'lag_{lag}'] = c['went_up'].shift(lag)
    
    c = c.dropna()
    
    balance = 1000
    history = [balance]
    stake = base_stake
    consec = 0
    wins = 0
    losses = 0
    busts = 0
    skips = 0
    skip_remaining = 0
    
    for _, row in c.iterrows():
        # Skip after loss cooldown
        if skip_remaining > 0:
            skip_remaining -= 1
            skips += 1
            continue
        
        # Check if all lags agree
        lags_agree = True
        direction = int(row['lag_1'])
        for lag in range(2, require_lags + 1):
            if int(row[f'lag_{lag}']) != direction:
                lags_agree = False
                break
        
        if not lags_agree:
            skips += 1
            continue
        
        # Martingale safety
        if consec >= max_losses or stake > max_exposure:
            busts += 1
            stake = base_stake
            consec = 0
        
        if stake > balance:
            break
        
        # Place trade following lag 1 direction
        actual = int(row['went_up'])
        bet = direction
        
        if bet == actual:
            balance += stake * payout
            wins += 1
            stake = base_stake
            consec = 0
        else:
            balance -= stake
            losses += 1
            consec += 1
            stake = round((stake + base_stake) / payout, 2)
            skip_remaining = skip_after_loss  # Activate cooldown
        
        history.append(balance)
    
    total = wins + losses
    return {
        'balance': balance,
        'profit': balance - 1000,
        'wins': wins,
        'losses': losses,
        'skips': skips,
        'trades': total,
        'win_rate': wins / total if total > 0 else 0,
        'busts': busts,
        'max_dd': 1000 - min(history),
        'history': history,
    }


# Test ALL combinations
configs = []
for lags in [1, 2, 3]:
    for skip in [0, 1, 2, 3]:
        label = f"Lags={lags}, Skip={skip}"
        r = combined_backtest(df, entry_second=0, require_lags=lags, skip_after_loss=skip)
        r['label'] = label
        r['lags'] = lags
        r['skip'] = skip
        configs.append(r)

# Sort by profit
configs.sort(key=lambda x: x['profit'], reverse=True)

print(f"COMBINED STRATEGY BACKTEST — All Lag + Skip Combinations")
print(f"Entry at :00 | $1 stake | 85% payout | 8-level martingale")
print(f"{'='*90}")
print(f"{'Config':<25} {'WinRate':>8} {'Trades':>7} {'Skips':>7} {'Busts':>6} {'Profit':>10} {'MaxDD':>8}")
print(f"{'-'*90}")
for r in configs:
    marker = " ★" if r == configs[0] else ""
    print(f"{r['label']:<25} {r['win_rate']:>7.1%} {r['trades']:>7} {r['skips']:>7} "
          f"{r['busts']:>6} ${r['profit']:>9.2f} ${r['max_dd']:>7.2f}{marker}")

# Plot top 5 vs baseline
fig, ax = plt.subplots(figsize=(16, 7))

# Baseline (lags=1, skip=0)
baseline_r = [r for r in configs if r['lags'] == 1 and r['skip'] == 0][0]
ax.plot(baseline_r['history'], label=f"Baseline L=1,S=0 (${baseline_r['profit']:.0f})", 
        linewidth=1, color='gray', linestyle='--')

# Top 5
for i, r in enumerate(configs[:5]):
    if r != baseline_r:
        ax.plot(r['history'], label=f"{r['label']} (${r['profit']:.0f})", linewidth=1.5)

ax.axhline(y=1000, color='yellow', linestyle='--', alpha=0.3)
ax.set_title('Combined Strategies — Martingale Equity Curves')
ax.set_xlabel('Trade #')
ax.set_ylabel('Balance ($)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Best
best = configs[0]
print(f"\nBEST OVERALL: {best['label']}")
print(f"  Win rate:  {best['win_rate']:.1%}")
print(f"  Profit:    ${best['profit']:.2f}")
print(f"  Trades:    {best['trades']} (skipped {best['skips']})")
print(f"  Busts:     {best['busts']}")
print(f"  Max DD:    ${best['max_dd']:.2f}")
print(f"  Per trade: ${best['profit']/best['trades']:.3f}")
hours = len(df) * 5 / 3600
print(f"  Per hour:  ${best['profit']/hours:.2f}")

# Compare to baseline
print(f"\n  vs Baseline: {best['win_rate'] - baseline_r['win_rate']:+.1%} win rate, "
      f"${best['profit'] - baseline_r['profit']:+.2f} profit")

## 9. Analyzing the 13% Losses — What Causes Momentum Breaks?

The 87% wins are momentum continuing. The 13% losses are momentum breaking. If we can identify the break conditions, we skip them and push toward 95%.

In [ ]:
# Build rich feature set at :00 entry point
# For each trade, capture everything we can SEE at the moment of entry

c = df[df['second'] == 0][['close']].copy()
c['future_close'] = c['close'].shift(-12)  # Result 60s later
c['went_up'] = (c['future_close'] > c['close']).astype(int)
c['prev_went_up'] = c['went_up'].shift(1)
c['matched'] = (c['went_up'] == c['prev_went_up']).astype(int)

# ── Feature 1: Size of last move (absolute) ──
c['last_move_abs'] = (c['future_close'].shift(1) - c['close'].shift(1)).abs()

# ── Feature 2: Price change in last 10 seconds before entry (:50 to :00) ──
c50 = df[df['second'] == 50][['close']].rename(columns={'close': 'p50'})
c55 = df[df['second'] == 55][['close']].rename(columns={'close': 'p55'})

c['p50'] = None
c['p55'] = None
for idx in c.index:
    t50 = idx - pd.Timedelta(seconds=10)
    t55 = idx - pd.Timedelta(seconds=5)
    m50 = c50.index.get_indexer([t50], method='nearest')
    m55 = c55.index.get_indexer([t55], method='nearest')
    if m50[0] >= 0: c.loc[idx, 'p50'] = c50.iloc[m50[0]]['p50']
    if m55[0] >= 0: c.loc[idx, 'p55'] = c55.iloc[m55[0]]['p55']

c['p50'] = c['p50'].astype(float)
c['p55'] = c['p55'].astype(float)
c['last_10s_change'] = c['close'] - c['p50']
c['last_5s_change'] = c['close'] - c['p55']

# Is the last 10s move in SAME direction as the previous result?
c['last_10s_confirms'] = (
    ((c['last_10s_change'] > 0) & (c['prev_went_up'] == 1)) |
    ((c['last_10s_change'] < 0) & (c['prev_went_up'] == 0))
).astype(int)

# ── Feature 3: Consecutive same-direction results (momentum age) ──
consec_same = []
count = 0
prev = None
for val in c['went_up'].values:
    if val == prev:
        count += 1
    else:
        count = 1
    consec_same.append(count)
    prev = val
c['momentum_age'] = pd.Series(consec_same, index=c.index).shift(1)

# ── Feature 4: Candle body ratio of last trade ──
# How much of the move was "body" vs "wick"
c['last_body'] = (c['future_close'].shift(1) - c['close'].shift(1)).abs()
# We need high/low for the last minute — approximate from nearby 5s candles
c['last_range'] = c['last_move_abs']  # Simplified: body ≈ range for now

c = c.dropna()
print(f"Trades with features: {len(c):,}")
print(f"Baseline win rate: {c['matched'].mean():.1%}")
print(f"Losses to analyze: {(c['matched'] == 0).sum()}")

# ═══════════════════════════════════════════════════════════
# ANALYSIS 1: Do the last 10 seconds predict momentum break?
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("ANALYSIS 1: Last 10 seconds before entry")
print("If price is already reversing from :50 to :00, momentum may break")
print(f"{'='*60}")

# When last 10s confirms momentum direction vs contradicts it
confirms = c[c['last_10s_confirms'] == 1]
contradicts = c[c['last_10s_confirms'] == 0]

wr_confirms = confirms['matched'].mean()
wr_contradicts = contradicts['matched'].mean()

print(f"  Last 10s CONFIRMS momentum:    {wr_confirms:.1%} ({len(confirms)} trades)")
print(f"  Last 10s CONTRADICTS momentum: {wr_contradicts:.1%} ({len(contradicts)} trades)")
print(f"  Difference: {wr_confirms - wr_contradicts:+.1%}")

if wr_confirms > wr_contradicts:
    print(f"\n  → Skip when last 10s contradicts: {wr_confirms:.1%} on {len(confirms)} trades")

# ═══════════════════════════════════════════════════════════
# ANALYSIS 2: Momentum age — how long has the trend been going?
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("ANALYSIS 2: Momentum age — minutes of same direction")
print(f"{'='*60}")

by_age = c.groupby('momentum_age').agg(
    win_rate=('matched', 'mean'),
    count=('matched', 'count'),
).reset_index()
by_age = by_age[by_age['count'] >= 20]

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['lime' if w > baseline_wr else 'red' for w in by_age['win_rate']]
ax.bar(by_age['momentum_age'], by_age['win_rate'], color=colors, alpha=0.7)
ax.axhline(y=baseline_wr, color='yellow', linestyle='--', label=f'Baseline ({baseline_wr:.1%})')
ax.set_title('Win Rate by Momentum Age (consecutive same-direction minutes)')
ax.set_xlabel('Minutes of sustained momentum')
ax.set_ylabel('Win Rate')
ax.legend()
plt.tight_layout()
plt.show()

for _, row in by_age.iterrows():
    marker = " ★" if row['win_rate'] > baseline_wr else ""
    print(f"  Age {int(row['momentum_age']):>3} min: {row['win_rate']:.1%} ({int(row['count'])} trades){marker}")

# ═══════════════════════════════════════════════════════════
# ANALYSIS 3: Move size quantiles — breakdowns within the 87%
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("ANALYSIS 3: Last move size — which quintile has the 13% losses?")
print(f"{'='*60}")

c['move_quintile'] = pd.qcut(c['last_move_abs'], q=5, 
                               labels=['Tiny', 'Small', 'Medium', 'Large', 'Huge'],
                               duplicates='drop')

by_move = c.groupby('move_quintile').agg(
    win_rate=('matched', 'mean'),
    count=('matched', 'count'),
    loss_pct=('matched', lambda x: (1-x).mean()),
).reset_index()

print(f"  {'Size':<10} {'WinRate':>8} {'Losses':>8} {'Trades':>8}")
print(f"  {'-'*40}")
for _, row in by_move.iterrows():
    losses = int(row['count'] * row['loss_pct'])
    print(f"  {row['move_quintile']:<10} {row['win_rate']:>7.1%} {losses:>8} {int(row['count']):>8}")

# ═══════════════════════════════════════════════════════════
# ANALYSIS 4: Combine best filters
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("ANALYSIS 4: Combined filters — skip the worst conditions")
print(f"{'='*60}")

combos = []

# Baseline
combos.append(('Baseline', c['matched'].mean(), len(c)))

# Skip tiny moves
mask = c['move_quintile'] != 'Tiny'
combos.append(('Skip tiny moves', c.loc[mask, 'matched'].mean(), mask.sum()))

# Skip when last 10s contradicts
mask = c['last_10s_confirms'] == 1
combos.append(('Skip contradicting 10s', c.loc[mask, 'matched'].mean(), mask.sum()))

# Skip tiny + contradicting
mask = (c['move_quintile'] != 'Tiny') & (c['last_10s_confirms'] == 1)
combos.append(('Skip tiny + contradicting', c.loc[mask, 'matched'].mean(), mask.sum()))

# Skip momentum age 1 (fresh reversals)
mask = c['momentum_age'] > 1
if mask.sum() > 100:
    combos.append(('Skip age=1 (fresh reversal)', c.loc[mask, 'matched'].mean(), mask.sum()))

# Only huge moves + confirms
mask = (c['move_quintile'] == 'Huge') & (c['last_10s_confirms'] == 1)
if mask.sum() > 50:
    combos.append(('Huge + confirms only', c.loc[mask, 'matched'].mean(), mask.sum()))

# Only Large+Huge + confirms
mask = c['move_quintile'].isin(['Large', 'Huge']) & (c['last_10s_confirms'] == 1)
if mask.sum() > 50:
    combos.append(('Large/Huge + confirms', c.loc[mask, 'matched'].mean(), mask.sum()))

# Skip tiny + skip age=1
mask = (c['move_quintile'] != 'Tiny') & (c['momentum_age'] > 1)
if mask.sum() > 100:
    combos.append(('Skip tiny + skip age=1', c.loc[mask, 'matched'].mean(), mask.sum()))

# Triple: skip tiny + confirms + age>1
mask = (c['move_quintile'] != 'Tiny') & (c['last_10s_confirms'] == 1) & (c['momentum_age'] > 1)
if mask.sum() > 50:
    combos.append(('Skip tiny + confirms + age>1', c.loc[mask, 'matched'].mean(), mask.sum()))

combos.sort(key=lambda x: x[1], reverse=True)
print(f"\n  {'Filter':<35} {'WinRate':>8} {'Trades':>8} {'vs Base':>8}")
print(f"  {'-'*60}")
for name, wr, trades in combos:
    diff = wr - baseline_wr
    marker = " ★" if wr > baseline_wr + 0.01 else ""
    print(f"  {name:<35} {wr:>7.1%} {trades:>8} {diff:>+7.1%}{marker}")

# ═══════════════════════════════════════════════════════════
# FULL MARTINGALE BACKTEST of best combined filter
# ═══════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("MARTINGALE BACKTEST — Best filter vs Baseline")
print(f"{'='*60}")

best_filter_name, best_filter_wr, _ = combos[0]

# Need to run proper simulation for best filter
# For now, compare the simple metrics
for name, wr, trades in [combos[0], ('Baseline', baseline_wr, len(c))]:
    profit_per_trade = wr * payout - (1 - wr) * 1.0 if 'payout' in dir() else wr * 0.85 - (1 - wr)
    total_profit = trades * profit_per_trade
    print(f"\n  {name}:")
    print(f"    Win rate:        {wr:.1%}")
    print(f"    Trades:          {trades}")
    print(f"    Profit/trade:    ${profit_per_trade:.4f}")
    print(f"    Est total profit: ${total_profit:.2f}")

## 10. Profit/Hour Comparison — Which Strategy Actually Makes the Most Money?

Higher win rate ≠ more profit if you skip too many trades. Let's find the sweet spot.

In [ ]:
# Full martingale backtest for each strategy variant
# Measure: win rate, skip %, profit, profit/hour, busts

hours = len(df) * 5 / 3600  # Total hours of data

def backtest_with_filter(df, entry_second, min_move=0.0, min_age=0, label=""):
    candles_ahead = (60 - entry_second) // 5
    c = df[df['second'] == entry_second][['close']].copy()
    c['future_close'] = c['close'].shift(-candles_ahead)
    c['went_up'] = (c['future_close'] > c['close']).astype(int)
    c['prev_went_up'] = c['went_up'].shift(1)
    c['prev_move'] = (c['future_close'].shift(1) - c['close'].shift(1)).abs()
    
    # Momentum age
    consec = []
    count = 0
    prev = None
    for val in c['went_up'].values:
        if val == prev: count += 1
        else: count = 1
        consec.append(count)
        prev = val
    c['mom_age'] = pd.Series(consec, index=c.index).shift(1)
    c = c.dropna()
    
    balance = 1000
    history = [balance]
    stake = 1.0
    cl = 0
    wins = 0; losses = 0; busts = 0; skips = 0
    last_dir = None
    
    for _, row in c.iterrows():
        # Filters
        skip = False
        if min_move > 0 and row['prev_move'] < min_move:
            skip = True
        if min_age > 0 and row['mom_age'] < min_age:
            skip = True
        
        if skip:
            skips += 1
            last_dir = int(row['went_up'])  # Still track direction
            continue
        
        if last_dir is None:
            bet = random.randint(0, 1)
        else:
            bet = last_dir
        
        actual = int(row['went_up'])
        
        if cl >= 8 or stake > 200:
            busts += 1; stake = 1.0; cl = 0
        if stake > balance: break
        
        if bet == actual:
            balance += stake * 0.85
            wins += 1; stake = 1.0; cl = 0
        else:
            balance -= stake
            losses += 1; cl += 1
            stake = round((stake + 1.0) / 0.85, 2)
        
        last_dir = actual
        history.append(balance)
    
    total = wins + losses
    opportunities = total + skips
    skip_pct = skips / opportunities * 100 if opportunities > 0 else 0
    wr = wins / total * 100 if total > 0 else 0
    profit = balance - 1000
    profit_per_hour = profit / hours if hours > 0 else 0
    
    return {
        'label': label,
        'win_rate': wr,
        'trades': total,
        'skips': skips,
        'skip_pct': skip_pct,
        'busts': busts,
        'profit': profit,
        'profit_per_hour': profit_per_hour,
        'max_dd': 1000 - min(history),
        'history': history,
    }

# Test all variants
variants = [
    {'label': 'v1: Entry :30, no filter', 'entry': 30, 'min_move': 0, 'min_age': 0},
    {'label': 'v2: Entry :00, no filter', 'entry': 0, 'min_move': 0, 'min_age': 0},
    {'label': 'Skip tiny only (>0.00019)', 'entry': 0, 'min_move': 0.00019, 'min_age': 0},
    {'label': 'Skip tiny+small (>0.00038)', 'entry': 0, 'min_move': 0.00038, 'min_age': 0},
    {'label': 'v3: Skip tiny + age>1', 'entry': 0, 'min_move': 0.00019, 'min_age': 2},
    {'label': 'Skip age>1 only', 'entry': 0, 'min_move': 0, 'min_age': 2},
    {'label': 'Skip age>2 only', 'entry': 0, 'min_move': 0, 'min_age': 3},
    {'label': 'v4: Large only (>0.00058)', 'entry': 0, 'min_move': 0.00058, 'min_age': 0},
    {'label': 'Medium+ (>0.00038) + age>1', 'entry': 0, 'min_move': 0.00038, 'min_age': 2},
]

results = []
for v in variants:
    r = backtest_with_filter(df, v['entry'], v['min_move'], v['min_age'], v['label'])
    results.append(r)

# Sort by profit/hour
results.sort(key=lambda x: x['profit_per_hour'], reverse=True)

print(f"STRATEGY RANKING BY PROFIT/HOUR")
print(f"Data: {hours:.0f} hours | 8-level martingale | $1 stakes")
print(f"{'='*100}")
print(f"{'Strategy':<35} {'Win%':>5} {'Skip%':>6} {'Trades':>7} {'Busts':>6} {'Profit':>9} {'$/hour':>8} {'$/trade':>8}")
print(f"{'-'*100}")
for r in results:
    marker = " ★" if r == results[0] else ""
    print(f"{r['label']:<35} {r['win_rate']:>4.1f}% {r['skip_pct']:>5.1f}% {r['trades']:>7} "
          f"{r['busts']:>6} ${r['profit']:>8.2f} ${r['profit_per_hour']:>7.2f} "
          f"${r['profit']/r['trades'] if r['trades'] > 0 else 0:>7.3f}{marker}")

# Plot: Win Rate vs Profit/Hour scatter
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: win rate vs profit/hour
for r in results:
    color = 'lime' if r == results[0] else 'cyan'
    size = 150 if r == results[0] else 80
    axes[0].scatter(r['win_rate'], r['profit_per_hour'], s=size, color=color, alpha=0.8)
    axes[0].annotate(r['label'].split(':')[-1].strip()[:20], 
                     (r['win_rate'], r['profit_per_hour']),
                     fontsize=7, color='white', ha='center', va='bottom')
axes[0].set_xlabel('Win Rate (%)')
axes[0].set_ylabel('Profit per Hour ($)')
axes[0].set_title('Win Rate vs Profit/Hour — The Sweet Spot')

# Equity curves for top 3 + baseline
for i, r in enumerate(results[:3]):
    axes[1].plot(r['history'], label=f"{r['label']} (${r['profit_per_hour']:.1f}/hr)", linewidth=1.5)
baseline = [x for x in results if 'v2' in x['label']][0]
if baseline not in results[:3]:
    axes[1].plot(baseline['history'], label=f"v2 baseline (${baseline['profit_per_hour']:.1f}/hr)", 
                 linewidth=1, linestyle='--', color='gray')
axes[1].axhline(y=1000, color='yellow', linestyle='--', alpha=0.3)
axes[1].set_title('Top 3 by $/hour — Equity Curves')
axes[1].set_xlabel('Trade #')
axes[1].set_ylabel('Balance ($)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

best = results[0]
print(f"\n★ BEST PROFIT/HOUR: {best['label']}")
print(f"  Win rate:      {best['win_rate']:.1f}%")
print(f"  Skip rate:     {best['skip_pct']:.1f}%")
print(f"  Profit/hour:   ${best['profit_per_hour']:.2f}")
print(f"  Total profit:  ${best['profit']:.2f}")
print(f"  Busts:         {best['busts']}")

## 11. Reverse Engineering the OTC Algorithm

Theory: the algorithm picks a trend direction and holds it for N minutes, then flips. Let's measure how long trends actually last.

In [ ]:
# Measure trend durations: how many consecutive minutes does price move in the same direction?
# Using :00 entry, :00 to :00 result windows

c = df[df['second'] == 0][['close']].copy()
c['future_close'] = c['close'].shift(-12)
c['went_up'] = (c['future_close'] > c['close']).astype(int)
c = c.dropna()

# Count consecutive same-direction runs (= trend duration)
trend_lengths = []
current_dir = None
current_len = 0

for val in c['went_up'].values:
    if val == current_dir:
        current_len += 1
    else:
        if current_len > 0:
            trend_lengths.append({'direction': 'UP' if current_dir == 1 else 'DOWN', 
                                   'length': current_len})
        current_dir = val
        current_len = 1

if current_len > 0:
    trend_lengths.append({'direction': 'UP' if current_dir == 1 else 'DOWN', 
                           'length': current_len})

trend_df = pd.DataFrame(trend_lengths)

print(f"TREND DURATION ANALYSIS")
print(f"{'='*60}")
print(f"Total trends detected: {len(trend_df)}")
print(f"Average trend length: {trend_df['length'].mean():.1f} minutes")
print(f"Median trend length: {trend_df['length'].median():.0f} minutes")
print(f"Max trend length: {trend_df['length'].max()} minutes")
print(f"Min trend length: {trend_df['length'].min()} minutes")

# Distribution of trend lengths
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram
axes[0][0].hist(trend_df['length'], bins=range(1, trend_df['length'].max() + 2), 
                color='cyan', alpha=0.7, edgecolor='white', linewidth=0.3)
axes[0][0].set_title('Distribution of Trend Durations')
axes[0][0].set_xlabel('Trend length (minutes)')
axes[0][0].set_ylabel('Count')
axes[0][0].axvline(x=trend_df['length'].mean(), color='yellow', linestyle='--', 
                    label=f'Mean: {trend_df["length"].mean():.1f} min')
axes[0][0].axvline(x=trend_df['length'].median(), color='lime', linestyle='--', 
                    label=f'Median: {trend_df["length"].median():.0f} min')
axes[0][0].legend()

# Cumulative: what % of trends last at least N minutes?
max_len = min(60, trend_df['length'].max())
survival = []
for n in range(1, max_len + 1):
    pct = (trend_df['length'] >= n).mean() * 100
    survival.append({'minutes': n, 'pct': pct})
survival_df = pd.DataFrame(survival)

axes[0][1].plot(survival_df['minutes'], survival_df['pct'], color='lime', linewidth=2)
axes[0][1].set_title('Trend Survival: % of trends lasting at least N minutes')
axes[0][1].set_xlabel('Minutes')
axes[0][1].set_ylabel('% of trends still alive')
axes[0][1].axhline(y=50, color='yellow', linestyle='--', label='50% survival')
median_survival = survival_df[survival_df['pct'] <= 50].iloc[0]['minutes'] if (survival_df['pct'] <= 50).any() else max_len
axes[0][1].axvline(x=median_survival, color='yellow', linestyle='--')
axes[0][1].legend()

# UP vs DOWN trend lengths
up_trends = trend_df[trend_df['direction'] == 'UP']['length']
down_trends = trend_df[trend_df['direction'] == 'DOWN']['length']

axes[1][0].hist([up_trends, down_trends], bins=range(1, 40), 
                color=['#4CAF50', '#F44336'], alpha=0.7, label=['UP trends', 'DOWN trends'])
axes[1][0].set_title('UP vs DOWN Trend Durations')
axes[1][0].set_xlabel('Trend length (minutes)')
axes[1][0].legend()

print(f"\nUP trends:   avg {up_trends.mean():.1f} min, median {up_trends.median():.0f} min ({len(up_trends)} trends)")
print(f"DOWN trends: avg {down_trends.mean():.1f} min, median {down_trends.median():.0f} min ({len(down_trends)} trends)")

# Conditional: given a trend has lasted N minutes, what's the probability it lasts 1 more?
conditional = []
for n in range(1, 30):
    lasted_n = trend_df[trend_df['length'] >= n]
    lasted_n_plus_1 = trend_df[trend_df['length'] >= n + 1]
    if len(lasted_n) > 20:
        prob = len(lasted_n_plus_1) / len(lasted_n)
        conditional.append({'age': n, 'prob_continue': prob, 'count': len(lasted_n)})

cond_df = pd.DataFrame(conditional)

axes[1][1].bar(cond_df['age'], cond_df['prob_continue'], color='cyan', alpha=0.7)
axes[1][1].axhline(y=0.5, color='red', linestyle='--', label='50% (coin flip)')
axes[1][1].set_title('P(trend continues 1 more minute | lasted N minutes)')
axes[1][1].set_xlabel('Trend age (minutes)')
axes[1][1].set_ylabel('P(continues)')
axes[1][1].set_ylim(0.4, 1.0)
axes[1][1].legend()

plt.tight_layout()
plt.show()

print(f"\nCONDITIONAL CONTINUATION PROBABILITY:")
print(f"{'Age':>5} {'P(continue)':>12} {'Trends':>8}")
print(f"{'-'*30}")
for _, row in cond_df.iterrows():
    marker = " ← peak" if row['prob_continue'] == cond_df['prob_continue'].max() else ""
    print(f"{int(row['age']):>5} {row['prob_continue']:>11.1%} {int(row['count']):>8}{marker}")

# Key insight
print(f"\nKEY INSIGHT:")
print(f"  Median trend: {trend_df['length'].median():.0f} minutes")
print(f"  Mean trend: {trend_df['length'].mean():.1f} minutes")
print(f"  50% survival at: ~{median_survival} minutes")
print(f"  Trend direction flips every ~{trend_df['length'].mean():.0f} minutes on average")

# Is the trend length distribution geometric (memoryless) or something else?
from scipy import stats
# Fit geometric distribution
p_flip = 1 / trend_df['length'].mean()  # Estimated flip probability per minute
print(f"\n  Estimated flip probability per minute: {p_flip:.3f} ({p_flip*100:.1f}%)")
print(f"  If geometric: P(continue) should be constant at {1-p_flip:.1%} regardless of age")
print(f"  Actual: check the bar chart — is it flat or does it change with age?")

## 12. Closing the Gap: 87% → 94% with Zero Skips

A competitor achieves **94% with no skips**. We're at 86.9% (v2, no skips). The 7% gap means they're correctly predicting ~half of our 13% losses.

**What could they know that we don't?**
1. **Better direction signal** — not just "follow last result" but something smarter
2. **Intra-minute price shape** — using the 12 candles within the minute, not just open/close
3. **Adaptive direction** — switching between "follow" and "fade" based on conditions
4. **Multi-timeframe** — combining minute-level and sub-minute signals
5. **Price level awareness** — absolute price zones where behavior changes

Let's test each systematically.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# REBUILD BASE DATA — :00 entry with rich features
# ═══════════════════════════════════════════════════════════════

c = df[df['second'] == 0][['close', 'open', 'high', 'low']].copy()
c['future_close'] = c['close'].shift(-12)  # Result 60s later
c['went_up'] = (c['future_close'] > c['close']).astype(int)
c['prev_went_up'] = c['went_up'].shift(1)
c['follow_correct'] = (c['went_up'] == c['prev_went_up']).astype(int)

# Previous move details
c['prev_move'] = c['future_close'].shift(1) - c['close'].shift(1)  # Signed
c['prev_move_abs'] = c['prev_move'].abs()

# Momentum age (consecutive same direction)
ages = []
count = 0
prev = None
for val in c['went_up'].values:
    if val == prev: count += 1
    else: count = 1
    ages.append(count)
    prev = val
c['momentum_age'] = pd.Series(ages, index=c.index).shift(1)

# Gather all 12 intra-minute candles for each minute
# Build features from the PREVIOUS minute's 12 candle ticks
for offset in range(0, 60, 5):
    sec_candles = df[df['second'] == offset][['close', 'open', 'high', 'low']].copy()
    suffix = f'_{offset:02d}'
    for col in ['close', 'open', 'high', 'low']:
        c[f'prev_{col}{suffix}'] = None

# This is slow with per-row lookups — use merge approach instead
# Build previous minute's intra-candle features more efficiently

# For each minute boundary at :00, the previous minute ran from :00(prev) to :55(prev)
# We need the 12 five-second candles: :00, :05, :10, ..., :55 of the PREVIOUS minute

# Get prices at key seconds of the PREVIOUS minute
for sec in [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55]:
    sec_df = df[df['second'] == sec][['close']].rename(columns={'close': f'pm_close_{sec:02d}'})
    # Shift by appropriate amount: sec=0 means 60s ago, sec=55 means 5s ago
    shift_seconds = 60 - sec
    for idx in c.index:
        target = idx - pd.Timedelta(seconds=shift_seconds)
        matches = sec_df.index.get_indexer([target], method='nearest')
        if matches[0] >= 0 and abs((sec_df.index[matches[0]] - target).total_seconds()) < 3:
            c.loc[idx, f'pm_close_{sec:02d}'] = sec_df.iloc[matches[0]][f'pm_close_{sec:02d}']

c = c.dropna(subset=['went_up', 'prev_went_up', 'momentum_age'])

# Convert pm columns to float
pm_cols = [f'pm_close_{s:02d}' for s in range(0, 60, 5)]
for col in pm_cols:
    c[col] = pd.to_numeric(c[col], errors='coerce')

c_full = c.dropna(subset=pm_cols)

print(f"Full dataset: {len(c_full):,} minutes with all intra-minute prices")
print(f"Baseline (follow last): {c_full['follow_correct'].mean():.1%}")
print(f"Losses to explain: {(c_full['follow_correct'] == 0).sum()}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# IDEA 1: Intra-minute price SHAPE of the previous minute
# ═══════════════════════════════════════════════════════════════
# The previous minute went UP or DOWN, but HOW did it get there?
# - Steady climb vs late spike
# - Reversal pattern (went up then came back)
# - Acceleration vs deceleration

print("IDEA 1: Previous Minute's Price Shape")
print("=" * 60)

# Build shape features from the 12 intra-minute prices
pm_prices = c_full[pm_cols].values  # shape: (N, 12)
pm_open = pm_prices[:, 0]   # Price at :00 (start of prev minute)
pm_close = pm_prices[:, -1]  # Price at :55 (end of prev minute)
pm_mid = pm_prices[:, 6]     # Price at :30 (midpoint)

# Feature: First half vs second half movement
first_half_move = pm_mid - pm_open      # :00 to :30
second_half_move = pm_close - pm_mid    # :30 to :55
total_move = pm_close - pm_open         # Full minute

c_full = c_full.copy()
c_full['first_half'] = first_half_move
c_full['second_half'] = second_half_move
c_full['total_move'] = total_move

# Did price accelerate or decelerate?
c_full['accelerating'] = (
    (c_full['second_half'].abs() > c_full['first_half'].abs())
).astype(int)

# Did price reverse in second half? (first half UP, second half DOWN or vice versa)
c_full['reversed'] = (
    (c_full['first_half'] * c_full['second_half']) < 0
).astype(int)

# How much of the move happened in last 10 seconds?
last_10s = pm_prices[:, -1] - pm_prices[:, -3]  # :50 to :55
c_full['late_push'] = last_10s
c_full['late_push_pct'] = np.where(
    np.abs(total_move) > 1e-8,
    last_10s / total_move,
    0
)

# Path efficiency: how straight was the path?
# Max deviation from straight line
straight_line = np.linspace(pm_open, pm_close, 12).T
deviations = np.abs(pm_prices - straight_line)
c_full['path_deviation'] = deviations.mean(axis=1)

# Monotonicity: how many of the 11 steps were in the final direction?
diffs = np.diff(pm_prices, axis=1)  # 11 step changes
final_up = (total_move > 0).astype(int)
steps_with_trend = np.where(
    final_up[:, None] == 1,
    (diffs > 0).sum(axis=1),
    (diffs < 0).sum(axis=1)
)
c_full['monotonicity'] = steps_with_trend / 11  # 1.0 = perfectly monotonic

# Now test: does shape predict whether "follow" works?
print("\n--- Win rate by shape features ---\n")

# Accelerating vs decelerating
for label, mask in [('Accelerating', c_full['accelerating'] == 1), 
                     ('Decelerating', c_full['accelerating'] == 0)]:
    wr = c_full.loc[mask, 'follow_correct'].mean()
    n = mask.sum()
    print(f"  {label:>15}: {wr:.1%} ({n} trades)")

print()

# Reversed vs continued
for label, mask in [('Continued (no reversal)', c_full['reversed'] == 0), 
                     ('Reversed mid-minute', c_full['reversed'] == 1)]:
    wr = c_full.loc[mask, 'follow_correct'].mean()
    n = mask.sum()
    print(f"  {label:>25}: {wr:.1%} ({n} trades)")

print()

# Monotonicity bins
c_full['mono_bin'] = pd.cut(c_full['monotonicity'], bins=[0, 0.4, 0.6, 0.8, 1.01],
                             labels=['Choppy (<40%)', 'Mixed (40-60%)', 'Smooth (60-80%)', 'Very smooth (80%+)'])
by_mono = c_full.groupby('mono_bin').agg(
    win_rate=('follow_correct', 'mean'),
    count=('follow_correct', 'count'),
).reset_index()
for _, row in by_mono.iterrows():
    print(f"  {row['mono_bin']:>20}: {row['win_rate']:.1%} ({int(row['count'])} trades)")

print()

# Late push — did price surge in last 10 seconds?
c_full['late_confirms'] = (
    ((c_full['late_push'] > 0) & (c_full['prev_went_up'] == 1)) |
    ((c_full['late_push'] < 0) & (c_full['prev_went_up'] == 0))
).astype(int)

for label, mask in [('Late push CONFIRMS trend', c_full['late_confirms'] == 1),
                     ('Late push OPPOSES trend', c_full['late_confirms'] == 0)]:
    wr = c_full.loc[mask, 'follow_correct'].mean()
    n = mask.sum()
    print(f"  {label:>30}: {wr:.1%} ({n} trades)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# IDEA 2: Adaptive Follow vs Fade
# ═══════════════════════════════════════════════════════════════
# What if SOMETIMES the right move is to FADE (go opposite)?
# If we could detect when to follow vs fade, we'd approach 100%.

print("IDEA 2: When Does 'Fade' Beat 'Follow'?")
print("=" * 60)

# For each trade, compute: would FOLLOW win? would FADE win?
c_full['fade_correct'] = 1 - c_full['follow_correct']

# When does fade work? Group by various conditions
print("\n--- Conditions where FADE might beat FOLLOW ---\n")

# By momentum age
print("By momentum age:")
by_age = c_full.groupby('momentum_age').agg(
    follow_wr=('follow_correct', 'mean'),
    fade_wr=('fade_correct', 'mean'),
    count=('follow_correct', 'count'),
).reset_index()
by_age = by_age[by_age['count'] >= 30]
for _, row in by_age.iterrows():
    better = "FOLLOW" if row['follow_wr'] > row['fade_wr'] else "FADE"
    marker = " ← fade wins!" if better == "FADE" else ""
    print(f"  Age {int(row['momentum_age']):>3}: Follow={row['follow_wr']:.1%}  Fade={row['fade_wr']:.1%}  "
          f"({int(row['count'])} trades) {marker}")

# By move size quintile
print("\nBy previous move size:")
c_full['move_q'] = pd.qcut(c_full['prev_move_abs'], q=5, 
                             labels=['Tiny', 'Small', 'Medium', 'Large', 'Huge'],
                             duplicates='drop')
by_move = c_full.groupby('move_q').agg(
    follow_wr=('follow_correct', 'mean'),
    fade_wr=('fade_correct', 'mean'),
    count=('follow_correct', 'count'),
).reset_index()
for _, row in by_move.iterrows():
    better = "FOLLOW" if row['follow_wr'] > row['fade_wr'] else "FADE"
    marker = " ← fade wins!" if better == "FADE" else ""
    print(f"  {row['move_q']:<10}: Follow={row['follow_wr']:.1%}  Fade={row['fade_wr']:.1%}  "
          f"({int(row['count'])} trades) {marker}")

# Combine: age=1 AND tiny move — is this where fading works?
print("\nCross-analysis: Age × Move Size (looking for fade zones):")
for age_min, age_max, age_label in [(1, 1, 'Age=1'), (2, 3, 'Age 2-3'), (4, 99, 'Age 4+')]:
    for move_label in ['Tiny', 'Small', 'Medium', 'Large', 'Huge']:
        mask = (c_full['momentum_age'] >= age_min) & (c_full['momentum_age'] <= age_max) & (c_full['move_q'] == move_label)
        if mask.sum() >= 20:
            follow_wr = c_full.loc[mask, 'follow_correct'].mean()
            n = mask.sum()
            better = "FOLLOW" if follow_wr > 0.5 else "FADE"
            marker = " ★ FADE ZONE" if follow_wr < 0.5 else ""
            print(f"  {age_label:>8} + {move_label:<8}: Follow={follow_wr:.1%} ({n} trades){marker}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# IDEA 3: The "Oracle" Strategy — What's the Theoretical Max?
# ═══════════════════════════════════════════════════════════════
# If we had a PERFECT predictor using only info available at entry time,
# what's the best we could do? This tells us if 94% is even possible.

print("IDEA 3: Oracle Analysis — What's Theoretically Possible?")
print("=" * 60)

# Build a feature matrix with everything we can observe at :00
features = c_full[['prev_move_abs', 'momentum_age', 'monotonicity',
                    'accelerating', 'reversed', 'late_confirms',
                    'first_half', 'second_half', 'path_deviation']].copy()

# Add: price position within recent range (is price near top or bottom?)
c_full['price_range_20'] = c_full['close'].rolling(20).max() - c_full['close'].rolling(20).min()
c_full['price_position'] = np.where(
    c_full['price_range_20'] > 1e-8,
    (c_full['close'] - c_full['close'].rolling(20).min()) / c_full['price_range_20'],
    0.5
)

# Add: direction of last 3, 5, 10 minute moves 
for lookback in [3, 5, 10]:
    c_full[f'up_ratio_{lookback}'] = c_full['prev_went_up'].rolling(lookback).mean()

# Use a Random Forest to find the theoretical ceiling
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

feature_cols = ['prev_move_abs', 'momentum_age', 'monotonicity',
                'accelerating', 'reversed', 'late_confirms',
                'first_half', 'second_half', 'path_deviation',
                'price_position']

# Add up_ratio features
for lookback in [3, 5, 10]:
    feature_cols.append(f'up_ratio_{lookback}')

c_ml = c_full.dropna(subset=feature_cols + ['went_up'])

X = c_ml[feature_cols].values
y = c_ml['went_up'].values

# Cross-validated accuracy
print("\nMachine learning ceiling test (5-fold CV):")
print("Can ML predict direction better than 'follow last'?\n")

# Always-follow baseline
follow_acc = c_ml['follow_correct'].mean()
print(f"  Follow Last Result:     {follow_acc:.1%} (our baseline)")

# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
rf_scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
print(f"  Random Forest:          {rf_scores.mean():.1%} (±{rf_scores.std():.1%})")

# Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
gb_scores = cross_val_score(gb, X, y, cv=5, scoring='accuracy')
print(f"  Gradient Boosting:      {gb_scores.mean():.1%} (±{gb_scores.std():.1%})")

# Train RF on full data to see feature importances
rf.fit(X, y)
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nFeature importances (Random Forest):")
for _, row in importances.iterrows():
    bar = '█' * int(row['importance'] * 100)
    print(f"  {row['feature']:<20} {row['importance']:.3f} {bar}")

# What if we use the ML model's prediction + follow as a hybrid?
# Train on first 70%, test on last 30%
split = int(len(c_ml) * 0.7)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

rf.fit(X_train, y_train)
ml_pred = rf.predict(X_test)
ml_acc = (ml_pred == y_test).mean()

follow_pred = c_ml.iloc[split:]['prev_went_up'].values.astype(int)
follow_acc_test = (follow_pred == y_test).mean()

# Hybrid: use ML when it disagrees with follow
agree_mask = ml_pred == follow_pred
disagree_mask = ~agree_mask

print(f"\nOut-of-sample test (last 30%):")
print(f"  Follow Last:  {follow_acc_test:.1%}")
print(f"  RF Model:     {ml_acc:.1%}")
print(f"  ML agrees with Follow: {agree_mask.mean():.1%} of the time")

if disagree_mask.sum() > 0:
    # When they disagree, who's right?
    ml_right_when_disagree = (ml_pred[disagree_mask] == y_test[disagree_mask]).mean()
    follow_right_when_disagree = (follow_pred[disagree_mask] == y_test[disagree_mask]).mean()
    print(f"\n  When ML disagrees with Follow ({disagree_mask.sum()} trades):")
    print(f"    ML correct:     {ml_right_when_disagree:.1%}")
    print(f"    Follow correct: {follow_right_when_disagree:.1%}")
    
    # Hybrid: follow when they agree, use ML when they disagree
    hybrid_pred = follow_pred.copy()
    hybrid_pred[disagree_mask] = ml_pred[disagree_mask]
    hybrid_acc = (hybrid_pred == y_test).mean()
    print(f"\n  Hybrid (follow when agree, ML when disagree): {hybrid_acc:.1%}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# IDEA 4: Trend Flip Detection — Predict WHEN the trend changes
# ═══════════════════════════════════════════════════════════════
# Our losses come from trend flips. If we can predict the flip,
# we can FADE on that one trade and get it right.
# 
# Key insight from trend analysis: trends get STRONGER with age.
# But what happens in the LAST minute before a flip?

print("IDEA 4: What Happens Right Before a Trend Flips?")
print("=" * 60)

# Rebuild trend data with pre-flip features
c_trend = df[df['second'] == 0][['close']].copy()
c_trend['future_close'] = c_trend['close'].shift(-12)
c_trend['went_up'] = (c_trend['future_close'] > c_trend['close']).astype(int)
c_trend['move_abs'] = (c_trend['future_close'] - c_trend['close']).abs()
c_trend = c_trend.dropna()

# Label: did the trend FLIP on the next minute?
c_trend['next_went_up'] = c_trend['went_up'].shift(-1)
c_trend['flipped'] = (c_trend['went_up'] != c_trend['next_went_up']).astype(int)

# Momentum age at this point
ages = []
count = 0
prev = None
for val in c_trend['went_up'].values:
    if val == prev: count += 1
    else: count = 1
    ages.append(count)
    prev = val
c_trend['age'] = ages

c_trend = c_trend.dropna()

print(f"\nOverall flip rate: {c_trend['flipped'].mean():.1%}")
print(f"(This is 1 - our follow win rate)\n")

# What predicts a flip?
# 1. Move size before flip
print("--- Move size in the minute BEFORE a flip ---")
flips = c_trend[c_trend['flipped'] == 1]
no_flips = c_trend[c_trend['flipped'] == 0]

print(f"  Avg move before FLIP:    {flips['move_abs'].mean():.6f}")
print(f"  Avg move before NO FLIP: {no_flips['move_abs'].mean():.6f}")
print(f"  Ratio: {flips['move_abs'].mean() / no_flips['move_abs'].mean():.2f}x")

# Move size distribution for flips vs non-flips
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
axes[0].hist(flips['move_abs'], bins=50, alpha=0.6, color='red', label='Before FLIP', density=True)
axes[0].hist(no_flips['move_abs'], bins=50, alpha=0.6, color='lime', label='Before CONTINUE', density=True)
axes[0].set_title('Move Size Distribution: Flips vs Continues')
axes[0].set_xlabel('Absolute price move')
axes[0].legend()
axes[0].set_xlim(0, 0.002)

# 2. Flip probability by move size decile
c_trend['move_decile'] = pd.qcut(c_trend['move_abs'], q=10, duplicates='drop')
by_decile = c_trend.groupby('move_decile').agg(
    flip_rate=('flipped', 'mean'),
    count=('flipped', 'count'),
).reset_index()

axes[1].bar(range(len(by_decile)), by_decile['flip_rate'], color='cyan', alpha=0.7)
axes[1].set_xticks(range(len(by_decile)))
axes[1].set_xticklabels([str(x)[:15] for x in by_decile['move_decile']], rotation=45, fontsize=7)
axes[1].set_title('Flip Probability by Move Size Decile')
axes[1].set_ylabel('P(flip)')
axes[1].axhline(y=c_trend['flipped'].mean(), color='yellow', linestyle='--', label='Average')
axes[1].legend()

# 3. Flip probability by trend age
by_age_flip = c_trend.groupby('age').agg(
    flip_rate=('flipped', 'mean'),
    count=('flipped', 'count'),
).reset_index()
by_age_flip = by_age_flip[by_age_flip['count'] >= 15]

axes[2].bar(by_age_flip['age'], by_age_flip['flip_rate'], color='orange', alpha=0.7)
axes[2].set_title('Flip Probability by Trend Age')
axes[2].set_xlabel('Trend age (minutes)')
axes[2].set_ylabel('P(flip on next minute)')
axes[2].axhline(y=c_trend['flipped'].mean(), color='yellow', linestyle='--', label='Average')
axes[2].legend()

plt.tight_layout()
plt.show()

# Key finding: does move size shrink before a flip?
# Compare last 3 moves before flip vs last 3 moves during trend
c_trend['prev_move_1'] = c_trend['move_abs'].shift(1)
c_trend['prev_move_2'] = c_trend['move_abs'].shift(2)
c_trend['prev_move_3'] = c_trend['move_abs'].shift(3)

# Average of last 3 moves
c_trend['avg_last_3'] = (c_trend['prev_move_1'] + c_trend['prev_move_2'] + c_trend['prev_move_3']) / 3

# Is current move smaller than average of last 3? (deceleration)
c_trend['decelerating'] = (c_trend['move_abs'] < c_trend['avg_last_3']).astype(int)

print("\n--- Deceleration as flip predictor ---")
for dec_label, dec_val in [('Decelerating', 1), ('Not decelerating', 0)]:
    mask = c_trend['decelerating'] == dec_val
    if mask.sum() > 50:
        flip_rate = c_trend.loc[mask, 'flipped'].mean()
        follow_wr = 1 - flip_rate
        n = mask.sum()
        print(f"  {dec_label:>20}: flip={flip_rate:.1%}  follow_wr={follow_wr:.1%}  ({n} trades)")

# Combination: age + deceleration
print("\n--- Age × Deceleration ---")
for age_min, age_max, age_label in [(1, 1, 'Age=1'), (2, 3, 'Age 2-3'), (4, 99, 'Age 4+')]:
    for dec_val, dec_label in [(1, 'Decel'), (0, 'Accel')]:
        mask = (c_trend['age'] >= age_min) & (c_trend['age'] <= age_max) & (c_trend['decelerating'] == dec_val)
        if mask.sum() >= 20:
            flip_rate = c_trend.loc[mask, 'flipped'].mean()
            follow_wr = 1 - flip_rate
            n = mask.sum()
            marker = " ★ FADE?" if flip_rate > 0.5 else ""
            print(f"  {age_label:>8} + {dec_label:>5}: flip={flip_rate:.1%}  follow={follow_wr:.1%}  ({n}){marker}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# IDEA 5: Adaptive Strategy — Follow OR Fade based on conditions
# ═══════════════════════════════════════════════════════════════
# Build the best no-skip strategy by choosing follow vs fade per trade

print("IDEA 5: Adaptive Strategy — Full Backtest")
print("=" * 60)

# Re-prepare data with all needed features
c_adapt = df[df['second'] == 0][['close']].copy()
c_adapt['future_close'] = c_adapt['close'].shift(-12)
c_adapt['went_up'] = (c_adapt['future_close'] > c_adapt['close']).astype(int)
c_adapt['prev_went_up'] = c_adapt['went_up'].shift(1)
c_adapt['move_abs'] = (c_adapt['future_close'] - c_adapt['close']).abs()
c_adapt['prev_move'] = c_adapt['move_abs'].shift(1)

# Momentum age
ages = []
count = 0
prev = None
for val in c_adapt['went_up'].dropna().values:
    if val == prev: count += 1
    else: count = 1
    ages.append(count)
    prev = val
c_adapt.loc[c_adapt['went_up'].dropna().index, 'age'] = ages
c_adapt['age'] = c_adapt['age'].shift(1)

# Deceleration
c_adapt['avg_prev_3'] = c_adapt['move_abs'].rolling(3).mean().shift(1)
c_adapt['decel'] = (c_adapt['prev_move'] < c_adapt['avg_prev_3']).astype(int)

# Last 5 up ratio
c_adapt['up_ratio_5'] = c_adapt['prev_went_up'].rolling(5).mean()

c_adapt = c_adapt.dropna()

def adaptive_backtest(df_in, rules, label=""):
    """
    Backtest with adaptive follow/fade rules.
    rules: list of (condition_fn, action) where action is 'follow' or 'fade'
    Falls through to 'follow' if no rule matches.
    """
    balance = 1000
    history = [balance]
    stake = 1.0
    consec = 0
    wins = 0; losses = 0; busts = 0
    last_dir = None
    
    for idx, row in df_in.iterrows():
        if last_dir is None:
            last_dir = int(row['went_up'])
            continue
        
        # Determine action
        action = 'follow'  # default
        for cond_fn, act in rules:
            if cond_fn(row):
                action = act
                break
        
        if action == 'follow':
            bet = last_dir
        else:  # fade
            bet = 1 - last_dir
        
        actual = int(row['went_up'])
        
        if consec >= 8 or stake > 200:
            busts += 1; stake = 1.0; consec = 0
        if stake > balance: break
        
        if bet == actual:
            balance += stake * 0.85
            wins += 1; stake = 1.0; consec = 0
        else:
            balance -= stake
            losses += 1; consec += 1
            stake = round((stake + 1.0) / 0.85, 2)
        
        last_dir = actual
        history.append(balance)
    
    total = wins + losses
    hours = len(df) * 5 / 3600
    profit = balance - 1000
    return {
        'label': label,
        'win_rate': wins / total * 100 if total > 0 else 0,
        'trades': total,
        'busts': busts,
        'profit': profit,
        'per_hour': profit / hours if hours > 0 else 0,
        'max_dd': 1000 - min(history),
        'history': history,
    }

# Test various adaptive rules
strategies = []

# Baseline: always follow
r = adaptive_backtest(c_adapt, [], "Always Follow (v2)")
strategies.append(r)

# Rule 1: Fade when age=1 AND tiny move (trend just started, weak signal)
r = adaptive_backtest(c_adapt, [
    (lambda row: row['age'] == 1 and row['prev_move'] < c_adapt['prev_move'].quantile(0.2), 'fade'),
], "Fade: age=1 + tiny")
strategies.append(r)

# Rule 2: Fade when decelerating AND age=1
r = adaptive_backtest(c_adapt, [
    (lambda row: row['age'] == 1 and row['decel'] == 1, 'fade'),
], "Fade: age=1 + decel")
strategies.append(r)

# Rule 3: Fade when move was tiny (bottom 10%)
tiny_threshold = c_adapt['prev_move'].quantile(0.10)
r = adaptive_backtest(c_adapt, [
    (lambda row: row['prev_move'] < tiny_threshold, 'fade'),
], "Fade: bottom 10% move")
strategies.append(r)

# Rule 4: Fade when age=1 (any move size)
r = adaptive_backtest(c_adapt, [
    (lambda row: row['age'] == 1, 'fade'),
], "Fade: all age=1")
strategies.append(r)

# Rule 5: Follow but use opposite when up_ratio suggests mean reversion
r = adaptive_backtest(c_adapt, [
    (lambda row: row['up_ratio_5'] >= 1.0, 'fade'),  # 5 UPs in a row, expect down
    (lambda row: row['up_ratio_5'] <= 0.0, 'fade'),  # 5 DOWNs in a row, expect up
], "Fade: 5 same in a row")
strategies.append(r)

# Rule 6: Combine best signals
r = adaptive_backtest(c_adapt, [
    (lambda row: row['age'] == 1 and row['prev_move'] < c_adapt['prev_move'].quantile(0.15), 'fade'),
    (lambda row: row['decel'] == 1 and row['age'] == 1, 'fade'),
], "Fade: age1+tiny OR age1+decel")
strategies.append(r)

# Rule 7: Use last 30s direction instead of last result when they disagree
# (need to add this data)
# For now, test simple moving average crossover idea
r = adaptive_backtest(c_adapt, [
    (lambda row: row['age'] == 1 and row['prev_move'] < c_adapt['prev_move'].quantile(0.20) and row['decel'] == 1, 'fade'),
], "Fade: age1 + tiny + decel")
strategies.append(r)

# Sort by win rate
strategies.sort(key=lambda x: x['win_rate'], reverse=True)

print(f"\n{'Strategy':<30} {'Win%':>6} {'Trades':>7} {'Busts':>6} {'$/hour':>8} {'Profit':>9}")
print(f"{'-'*75}")
for s in strategies:
    marker = " ★" if s['win_rate'] == strategies[0]['win_rate'] else ""
    print(f"{s['label']:<30} {s['win_rate']:>5.1f}% {s['trades']:>7} {s['busts']:>6} "
          f"${s['per_hour']:>7.2f} ${s['profit']:>8.2f}{marker}")

# Plot top 3 vs baseline
fig, ax = plt.subplots(figsize=(16, 7))
baseline_s = [s for s in strategies if 'Always' in s['label']][0]
ax.plot(baseline_s['history'], label=f"Always Follow ({baseline_s['win_rate']:.1f}%)", 
        linewidth=1, color='gray', linestyle='--')
for s in strategies[:3]:
    if s != baseline_s:
        ax.plot(s['history'], label=f"{s['label']} ({s['win_rate']:.1f}%)", linewidth=1.5)
ax.axhline(y=1000, color='yellow', linestyle='--', alpha=0.3)
ax.set_title('Adaptive Strategies — Can We Beat Pure Follow?')
ax.set_xlabel('Trade #')
ax.set_ylabel('Balance ($)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

best = strategies[0]
print(f"\n★ BEST: {best['label']}")
print(f"  Win rate:    {best['win_rate']:.1f}%")
print(f"  Profit/hour: ${best['per_hour']:.2f}")
print(f"  vs baseline: {best['win_rate'] - baseline_s['win_rate']:+.1f}% win rate")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# IDEA 6: What if "Follow Last" is Wrong — Alternative Base Signals
# ═══════════════════════════════════════════════════════════════
# Maybe 94% comes from a completely different base signal, not
# "follow last result." Let's test every possible direction rule.

print("IDEA 6: Alternative Direction Signals (No Skips)")
print("=" * 60)

c_alt = df[df['second'] == 0][['close']].copy()
c_alt['future_close'] = c_alt['close'].shift(-12)
c_alt['went_up'] = (c_alt['future_close'] > c_alt['close']).astype(int)
c_alt = c_alt.dropna()

# Get prices at various seconds for direction signals
for sec in [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55]:
    sec_df = df[df['second'] == sec][['close']].rename(columns={'close': f'p_{sec:02d}'})
    shift_secs = 60 - sec
    for idx in c_alt.index:
        target = idx - pd.Timedelta(seconds=shift_secs)
        matches = sec_df.index.get_indexer([target], method='nearest')
        if matches[0] >= 0 and abs((sec_df.index[matches[0]] - target).total_seconds()) < 3:
            c_alt.loc[idx, f'p_{sec:02d}'] = sec_df.iloc[matches[0]][f'p_{sec:02d}']

# Convert to float
for sec in [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55]:
    c_alt[f'p_{sec:02d}'] = pd.to_numeric(c_alt[f'p_{sec:02d}'], errors='coerce')

c_alt_full = c_alt.dropna()

print(f"Dataset: {len(c_alt_full):,} minutes\n")

signals = []

# Signal 1: Follow last minute result (our baseline)
prev_up = c_alt_full['went_up'].shift(1)
acc = (c_alt_full['went_up'] == prev_up).dropna().mean()
signals.append(('Follow last result', acc))

# Signal 2: Follow last 30 seconds (:30 to :00)
last_30s_up = (c_alt_full['close'] > c_alt_full['p_30']).astype(int)
acc = (c_alt_full['went_up'] == last_30s_up).mean()
signals.append(('Follow :30→:00 direction', acc))

# Signal 3: Follow last 15 seconds (:45 to :00)
last_15s_up = (c_alt_full['close'] > c_alt_full['p_45']).astype(int)
acc = (c_alt_full['went_up'] == last_15s_up).mean()
signals.append(('Follow :45→:00 direction', acc))

# Signal 4: Follow last 10 seconds (:50 to :00)
last_10s_up = (c_alt_full['close'] > c_alt_full['p_50']).astype(int)
acc = (c_alt_full['went_up'] == last_10s_up).mean()
signals.append(('Follow :50→:00 direction', acc))

# Signal 5: Follow last 5 seconds (:55 to :00)
last_5s_up = (c_alt_full['close'] > c_alt_full['p_55']).astype(int)
acc = (c_alt_full['went_up'] == last_5s_up).mean()
signals.append(('Follow :55→:00 direction', acc))

# Signal 6: FADE last 30 seconds (mean reversion)
acc = (c_alt_full['went_up'] != last_30s_up).mean()
signals.append(('Fade :30→:00 direction', acc))

# Signal 7: Follow first half of prev minute (:00 to :30)
first_half_up = (c_alt_full['p_30'] > c_alt_full['p_05']).astype(int)  # approx
acc = (c_alt_full['went_up'] == first_half_up).mean()
signals.append(('Follow :05→:30 (first half)', acc))

# Signal 8: Follow the dominant direction of prev minute (majority of 5s candles)
prev_prices = c_alt_full[[f'p_{s:02d}' for s in [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55]]].values
diffs = np.diff(prev_prices, axis=1)
majority_up = (diffs > 0).sum(axis=1) > (diffs < 0).sum(axis=1)
acc = (c_alt_full['went_up'].values == majority_up.astype(int)).mean()
signals.append(('Follow majority of 5s steps', acc))

# Signal 9: Follow the direction of the LARGEST 5s move in prev minute
largest_step_idx = np.abs(diffs).argmax(axis=1)
largest_step_dir = np.array([diffs[i, j] > 0 for i, j in enumerate(largest_step_idx)]).astype(int)
acc = (c_alt_full['went_up'].values == largest_step_dir).mean()
signals.append(('Follow largest 5s step', acc))

# Signal 10: Follow the direction of the LAST non-zero 5s step
last_nonzero_dir = []
for row in diffs:
    nonzero = row[row != 0]
    if len(nonzero) > 0:
        last_nonzero_dir.append(int(nonzero[-1] > 0))
    else:
        last_nonzero_dir.append(0)
last_nonzero_dir = np.array(last_nonzero_dir)
acc = (c_alt_full['went_up'].values == last_nonzero_dir).mean()
signals.append(('Follow last non-zero 5s step', acc))

# Signal 11: Weighted: follow if last 3 results all same direction
prev1 = c_alt_full['went_up'].shift(1)
prev2 = c_alt_full['went_up'].shift(2)
prev3 = c_alt_full['went_up'].shift(3)
all_up = ((prev1 == 1) & (prev2 == 1) & (prev3 == 1))
all_down = ((prev1 == 0) & (prev2 == 0) & (prev3 == 0))
# When all 3 agree, follow; when not, use last result
signal_11 = prev1.copy()
# acc just on the "all agree" subset
agree_mask = all_up | all_down
if agree_mask.sum() > 0:
    acc_agree = (c_alt_full.loc[agree_mask.dropna().index[agree_mask.dropna()], 'went_up'] == 
                 prev1.loc[agree_mask.dropna().index[agree_mask.dropna()]]).mean()
    signals.append((f'Follow when last 3 agree ({agree_mask.sum()} trades)', acc_agree))

# Signal 12: Follow 2-minute-ago result (lag 2)
prev2_dir = c_alt_full['went_up'].shift(2)
acc = (c_alt_full['went_up'] == prev2_dir).dropna().mean()
signals.append(('Follow lag-2 result', acc))

# Sort and display
signals.sort(key=lambda x: x[1], reverse=True)

print(f"{'Signal':<40} {'Accuracy':>8}")
print(f"{'-'*50}")
for name, acc in signals:
    marker = " ★" if acc > 0.87 else ""
    print(f"{name:<40} {acc:>7.1%}{marker}")

# Visualize
fig, ax = plt.subplots(figsize=(14, 6))
names = [s[0][:30] for s in signals]
accs = [s[1] for s in signals]
colors = ['lime' if a > 0.87 else ('gold' if a > 0.54 else 'red') for a in accs]
ax.barh(range(len(signals)), accs, color=colors, alpha=0.7)
ax.axvline(x=0.869, color='yellow', linestyle='--', label='Follow Last (86.9%)')
ax.axvline(x=0.94, color='red', linestyle='--', label='Target (94%)')
ax.set_yticks(range(len(signals)))
ax.set_yticklabels(names, fontsize=8)
ax.set_title('Direction Signal Accuracy — Which Predicts Best?')
ax.set_xlabel('Accuracy')
ax.legend()
plt.tight_layout()
plt.show()

## 13. The Breakthrough: "Follow When Last 3 Agree" = 90%

**Key finding:** When the last 3 minute results all went the same direction, following that direction is **90.0% accurate** — on 77% of all trades.

**Path to 94% with zero skips:**
- 77% of trades: lags agree → follow → 90% ✓
- 23% of trades: lags disagree → **need a signal here**
- If we get the disagree trades ~80% right: blended = 0.77×0.90 + 0.23×0.80 = **87.7%**
- If we get them ~94% right: blended = 0.77×0.90 + 0.23×0.94 = **90.9%**
- To hit 94% overall: need disagree accuracy = (0.94 - 0.77×0.90) / 0.23 = **107%** ← impossible

Wait — that means 94% CANNOT come from "3 agree + something else." We need to improve the "agree" accuracy too, or find a completely different approach.

Let's dig deeper into both paths.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# DEEP DIVE: What happens when last 3 lags DISAGREE?
# And can we improve the AGREE accuracy above 90%?
# ═══════════════════════════════════════════════════════════════

print("DEEP DIVE: Disagree Trades & Improving Agree Accuracy")
print("=" * 60)

c_deep = df[df['second'] == 0][['close']].copy()
c_deep['future_close'] = c_deep['close'].shift(-12)
c_deep['went_up'] = (c_deep['future_close'] > c_deep['close']).astype(int)
c_deep['lag1'] = c_deep['went_up'].shift(1)
c_deep['lag2'] = c_deep['went_up'].shift(2)
c_deep['lag3'] = c_deep['went_up'].shift(3)
c_deep['lag4'] = c_deep['went_up'].shift(4)
c_deep['lag5'] = c_deep['went_up'].shift(5)
c_deep['move_abs'] = (c_deep['future_close'] - c_deep['close']).abs()
c_deep['prev_move'] = c_deep['move_abs'].shift(1)

# Momentum age
ages = []
count = 0
prev = None
for val in c_deep['went_up'].dropna().values:
    if val == prev: count += 1
    else: count = 1
    ages.append(count)
    prev = val
c_deep.loc[c_deep['went_up'].dropna().index, 'age'] = ages
c_deep['age'] = c_deep['age'].shift(1)

c_deep = c_deep.dropna()

# Split into agree (last 3 same) vs disagree
all_same = (c_deep['lag1'] == c_deep['lag2']) & (c_deep['lag2'] == c_deep['lag3'])
agree = c_deep[all_same]
disagree = c_deep[~all_same]

follow_wr_agree = (agree['went_up'] == agree['lag1']).mean()
follow_wr_disagree = (disagree['went_up'] == disagree['lag1']).mean()

print(f"\nSplit:")
print(f"  Agree (last 3 same):    {len(agree):,} trades ({len(agree)/len(c_deep)*100:.0f}%) → follow={follow_wr_agree:.1%}")
print(f"  Disagree (mixed):       {len(disagree):,} trades ({len(disagree)/len(c_deep)*100:.0f}%) → follow={follow_wr_disagree:.1%}")

# ── Part A: What's the pattern in DISAGREE trades? ──
print(f"\n{'='*60}")
print("PART A: Disagree trades — what patterns exist?")
print(f"{'='*60}\n")

# When lags disagree, what are the actual patterns?
# lag1, lag2, lag3 can be: UUD, UDU, DUU, DDU, DUD, UDD
disagree['pattern'] = disagree['lag1'].astype(str) + disagree['lag2'].astype(str) + disagree['lag3'].astype(str)
by_pattern = disagree.groupby('pattern').agg(
    follow_wr=('went_up', lambda x: (x == disagree.loc[x.index, 'lag1']).mean()),
    up_pct=('went_up', 'mean'),
    count=('went_up', 'count'),
).reset_index()

print(f"  {'Pattern':>10} {'Follow%':>8} {'Up%':>6} {'N':>6}  Meaning")
print(f"  {'-'*55}")
for _, row in by_pattern.iterrows():
    p = row['pattern']
    meaning = {
        '110': 'UP UP DOWN (reversal just happened)',
        '100': 'UP DOWN DOWN (2-min downtrend after UP)',
        '010': 'DOWN UP DOWN (bounce, back down)',
        '001': 'DOWN DOWN UP (reversal just happened)',
        '011': 'DOWN UP UP (2-min uptrend after DOWN)',
        '101': 'UP DOWN UP (bounce, back up)',
    }.get(p, '')
    print(f"  {p:>10} {row['follow_wr']:>7.1%} {row['up_pct']:>5.1%} {int(row['count']):>6}  {meaning}")

# What signal works best for disagree trades?
print(f"\n  Signals for disagree trades:")
# Follow lag1 (most recent)
acc1 = (disagree['went_up'] == disagree['lag1']).mean()
# Follow lag2
acc2 = (disagree['went_up'] == disagree['lag2']).mean()
# Follow lag3
acc3 = (disagree['went_up'] == disagree['lag3']).mean()
# Follow majority vote of lag1-3
majority = ((disagree['lag1'] + disagree['lag2'] + disagree['lag3']) >= 2).astype(int)
acc_maj = (disagree['went_up'] == majority).mean()
# Follow lag1 but FADE (go opposite)
acc_fade = (disagree['went_up'] != disagree['lag1']).mean()

print(f"    Follow lag1 (most recent): {acc1:.1%}")
print(f"    Follow lag2:               {acc2:.1%}")
print(f"    Follow lag3:               {acc3:.1%}")
print(f"    Majority vote (2 of 3):    {acc_maj:.1%}")
print(f"    Fade lag1:                 {acc_fade:.1%}")

# ── Part B: Can we improve AGREE accuracy above 90%? ──
print(f"\n{'='*60}")
print("PART B: Improving agree accuracy (currently 90%)")
print(f"{'='*60}\n")

# Within the agree trades, what sub-conditions give >90%?
# By momentum age
print("  Agree trades by momentum age:")
for age_val in sorted(agree['age'].unique()):
    if age_val <= 20:
        mask = agree['age'] == age_val
        if mask.sum() >= 20:
            wr = (agree.loc[mask, 'went_up'] == agree.loc[mask, 'lag1']).mean()
            n = mask.sum()
            marker = " ★" if wr > 0.92 else ""
            print(f"    Age {int(age_val):>3}: {wr:.1%} ({n} trades){marker}")

# By previous move size
print("\n  Agree trades by previous move size:")
agree_with_move = agree.copy()
agree_with_move['move_q'] = pd.qcut(agree_with_move['prev_move'], q=5, 
                                      labels=['Tiny', 'Small', 'Medium', 'Large', 'Huge'],
                                      duplicates='drop')
by_move = agree_with_move.groupby('move_q').apply(
    lambda g: (g['went_up'] == g['lag1']).mean()
).reset_index()
by_move.columns = ['move_q', 'follow_wr']
for _, row in by_move.iterrows():
    marker = " ★" if row['follow_wr'] > 0.92 else ""
    print(f"    {row['move_q']:<10}: {row['follow_wr']:.1%}{marker}")

# ── Part C: Extend to more lags — does 4 or 5 agree beat 3? ──
print(f"\n{'='*60}")
print("PART C: More lags — 4 agree, 5 agree")
print(f"{'='*60}\n")

for n_lags in [2, 3, 4, 5]:
    lag_cols = [f'lag{i}' for i in range(1, n_lags + 1)]
    all_match = c_deep[lag_cols[0]] == c_deep[lag_cols[1]]
    for col in lag_cols[2:]:
        all_match = all_match & (c_deep[lag_cols[0]] == c_deep[col])
    
    subset = c_deep[all_match]
    if len(subset) > 50:
        wr = (subset['went_up'] == subset['lag1']).mean()
        pct = len(subset) / len(c_deep) * 100
        print(f"  Last {n_lags} agree: {wr:.1%} on {len(subset):,} trades ({pct:.0f}%)")

# ── Part D: Blended strategy — what's our best no-skip accuracy? ──
print(f"\n{'='*60}")
print("PART D: Best blended no-skip strategy")
print(f"{'='*60}\n")

# Strategy: when last 3 agree → follow; when disagree → use best disagree signal
best_disagree_signal = 'majority'  # we'll use whatever scored highest above

# Build prediction
pred = pd.Series(index=c_deep.index, dtype=float)

# Agree: follow lag1
pred[all_same] = c_deep.loc[all_same, 'lag1']

# Disagree: try each signal
for signal_name, signal_fn in [
    ('follow lag1', lambda d: d['lag1']),
    ('follow lag2', lambda d: d['lag2']),
    ('majority vote', lambda d: ((d['lag1'] + d['lag2'] + d['lag3']) >= 2).astype(float)),
    ('fade lag1', lambda d: 1 - d['lag1']),
]:
    pred_test = pred.copy()
    pred_test[~all_same] = signal_fn(c_deep.loc[~all_same])
    pred_test = pred_test.dropna()
    acc = (c_deep.loc[pred_test.index, 'went_up'] == pred_test).mean()
    
    # Compare to pure follow
    pure_follow = (c_deep.loc[pred_test.index, 'went_up'] == c_deep.loc[pred_test.index, 'lag1']).mean()
    diff = acc - pure_follow
    marker = " ★" if acc > pure_follow else ""
    print(f"  Agree→follow, Disagree→{signal_name:<15}: {acc:.1%} (vs pure follow {pure_follow:.1%}, {diff:+.1%}){marker}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# IDEA 7: Pattern-Based Direction — Use the EXACT 3-lag Pattern
# ═══════════════════════════════════════════════════════════════
# Instead of just "agree vs disagree", learn the optimal direction 
# for EACH of the 8 possible 3-lag patterns (UUU, UUD, UDU, etc.)

print("IDEA 7: Pattern-Based Direction (Every Possible 3-Lag Combo)")
print("=" * 60)

# Build all 8 patterns
c_deep['pattern'] = (c_deep['lag1'].astype(int).astype(str) + 
                      c_deep['lag2'].astype(int).astype(str) + 
                      c_deep['lag3'].astype(int).astype(str))

by_pattern = c_deep.groupby('pattern').agg(
    up_pct=('went_up', 'mean'),
    follow_wr=('went_up', lambda x: (x == c_deep.loc[x.index, 'lag1']).mean()),
    count=('went_up', 'count'),
).reset_index()

# For each pattern, the BEST direction to bet
by_pattern['best_direction'] = np.where(by_pattern['up_pct'] > 0.5, 'UP', 'DOWN')
by_pattern['best_accuracy'] = np.where(
    by_pattern['up_pct'] > 0.5, by_pattern['up_pct'], 1 - by_pattern['up_pct']
)

print(f"\n  {'Pattern':>8} {'P(UP)':>7} {'Follow%':>8} {'Best':>6} {'Best%':>7} {'N':>6}")
print(f"  {'-'*50}")
for _, row in by_pattern.iterrows():
    p = row['pattern']
    desc = f"{'U' if p[0]=='1' else 'D'}{'U' if p[1]=='1' else 'D'}{'U' if p[2]=='1' else 'D'}"
    marker = " ★" if row['best_accuracy'] > 0.90 else ""
    print(f"  {desc:>8} {row['up_pct']:>6.1%} {row['follow_wr']:>7.1%} {row['best_direction']:>6} "
          f"{row['best_accuracy']:>6.1%} {int(row['count']):>6}{marker}")

# Build the OPTIMAL pattern-based strategy
# For each trade, look up the pattern and bet the historically best direction
pattern_map = dict(zip(by_pattern['pattern'], by_pattern['up_pct']))

c_deep['pattern_pred'] = c_deep['pattern'].map(lambda p: 1 if pattern_map.get(p, 0.5) > 0.5 else 0)
pattern_acc = (c_deep['went_up'] == c_deep['pattern_pred']).mean()

# Compare
follow_acc = (c_deep['went_up'] == c_deep['lag1']).mean()

print(f"\n  RESULTS:")
print(f"    Pure Follow Last:  {follow_acc:.1%}")
print(f"    Pattern-Based:     {pattern_acc:.1%}")
print(f"    Improvement:       {pattern_acc - follow_acc:+.1%}")

# ═══════════════════════════════════════════════════════════════
# Extend to 4-lag and 5-lag patterns (16 and 32 combos)
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("Extending to 4-lag and 5-lag patterns")
print(f"{'='*60}\n")

for n_lags, min_samples in [(3, 50), (4, 30), (5, 20)]:
    lag_cols = [f'lag{i}' for i in range(1, n_lags + 1)]
    c_deep[f'pattern_{n_lags}'] = c_deep[lag_cols].astype(int).astype(str).agg(''.join, axis=1)
    
    by_pat = c_deep.groupby(f'pattern_{n_lags}').agg(
        up_pct=('went_up', 'mean'),
        count=('went_up', 'count'),
    ).reset_index()
    
    # Only use patterns with enough samples
    by_pat_valid = by_pat[by_pat['count'] >= min_samples]
    pat_map = dict(zip(by_pat_valid[f'pattern_{n_lags}'], by_pat_valid['up_pct']))
    
    c_deep[f'pred_{n_lags}'] = c_deep[f'pattern_{n_lags}'].map(
        lambda p: 1 if pat_map.get(p, 0.5) > 0.5 else 0
    )
    acc = (c_deep['went_up'] == c_deep[f'pred_{n_lags}']).mean()
    
    print(f"  {n_lags}-lag patterns: {len(by_pat_valid)}/{len(by_pat)} usable patterns → {acc:.1%} accuracy")

# ═══════════════════════════════════════════════════════════════
# Out-of-sample validation — train on first 70%, test on last 30%
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("OUT-OF-SAMPLE: Train patterns on first 70%, test on last 30%")
print(f"{'='*60}\n")

split = int(len(c_deep) * 0.7)
train = c_deep.iloc[:split]
test = c_deep.iloc[split:]

for n_lags in [3, 4, 5]:
    # Learn pattern → direction mapping from training data
    pat_col = f'pattern_{n_lags}'
    train_map = train.groupby(pat_col)['went_up'].mean().to_dict()
    
    # Apply to test data
    test_pred = test[pat_col].map(lambda p: 1 if train_map.get(p, 0.5) > 0.5 else 0)
    oos_acc = (test['went_up'] == test_pred).mean()
    
    # Compare to follow-last on test
    follow_test = (test['went_up'] == test['lag1']).mean()
    
    print(f"  {n_lags}-lag pattern: OOS={oos_acc:.1%} vs Follow={follow_test:.1%} ({oos_acc-follow_test:+.1%})")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# IDEA 8: Pattern + Move Size — The Full Combo
# ═══════════════════════════════════════════════════════════════
# Combine lag patterns with move size for the ultimate no-skip strategy

print("IDEA 8: Pattern + Move Size Combined")
print("=" * 60)

# Categorize move size into 3 buckets (more granularity = less samples per bucket)
c_deep['move_cat'] = pd.qcut(c_deep['prev_move'], q=3, labels=['S', 'M', 'L'], duplicates='drop')

# Combined pattern: 3-lag pattern + move size = 8 × 3 = 24 combos
c_deep['combo'] = c_deep['pattern'] + '_' + c_deep['move_cat'].astype(str)

by_combo = c_deep.groupby('combo').agg(
    up_pct=('went_up', 'mean'),
    count=('went_up', 'count'),
).reset_index()

by_combo['best_dir'] = np.where(by_combo['up_pct'] > 0.5, 1, 0)
by_combo['best_acc'] = np.where(by_combo['up_pct'] > 0.5, by_combo['up_pct'], 1 - by_combo['up_pct'])

by_combo = by_combo.sort_values('best_acc', ascending=False)

print(f"\n  {'Combo':<12} {'P(UP)':>7} {'Best%':>7} {'N':>6}")
print(f"  {'-'*35}")
for _, row in by_combo.iterrows():
    marker = " ★" if row['best_acc'] > 0.92 else ""
    print(f"  {row['combo']:<12} {row['up_pct']:>6.1%} {row['best_acc']:>6.1%} {int(row['count']):>6}{marker}")

# Build optimal combo strategy
combo_map = dict(zip(by_combo['combo'], by_combo['best_dir']))
c_deep['combo_pred'] = c_deep['combo'].map(combo_map)
combo_acc = (c_deep['went_up'] == c_deep['combo_pred']).mean()

# Out-of-sample
train_combo = c_deep.iloc[:split]
test_combo = c_deep.iloc[split:]
train_combo_map = train_combo.groupby('combo')['went_up'].mean().to_dict()
test_combo_pred = test_combo['combo'].map(lambda p: 1 if train_combo_map.get(p, 0.5) > 0.5 else 0)
combo_oos = (test_combo['went_up'] == test_combo_pred).mean()
follow_oos = (test_combo['went_up'] == test_combo['lag1']).mean()

print(f"\n  RESULTS:")
print(f"    In-sample:      {combo_acc:.1%}")
print(f"    Out-of-sample:  {combo_oos:.1%} (vs Follow {follow_oos:.1%})")
print(f"    Improvement:    {combo_oos - follow_oos:+.1%}")

# ═══════════════════════════════════════════════════════════════
# FINAL: Full martingale backtest of best strategies
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("FULL MARTINGALE BACKTEST — Best No-Skip Strategies")
print(f"{'='*60}\n")

hours = len(df) * 5 / 3600

def pattern_backtest(df_in, pattern_map_fn, label=""):
    """Backtest using a pattern → direction lookup."""
    balance = 1000
    history = [balance]
    stake = 1.0
    consec = 0
    wins = 0; losses = 0; busts = 0
    
    for _, row in df_in.iterrows():
        # Get prediction from pattern
        bet = pattern_map_fn(row)
        if bet is None:
            continue
            
        actual = int(row['went_up'])
        
        if consec >= 8 or stake > 200:
            busts += 1; stake = 1.0; consec = 0
        if stake > balance: break
        
        if bet == actual:
            balance += stake * 0.85
            wins += 1; stake = 1.0; consec = 0
        else:
            balance -= stake
            losses += 1; consec += 1
            stake = round((stake + 1.0) / 0.85, 2)
        
        history.append(balance)
    
    total = wins + losses
    profit = balance - 1000
    return {
        'label': label,
        'win_rate': wins / total * 100 if total > 0 else 0,
        'trades': total,
        'busts': busts,
        'profit': profit,
        'per_hour': profit / hours if hours > 0 else 0,
        'history': history,
    }

# Use training data to learn patterns, test on ALL data (in-sample for now)
# Then do proper OOS split

# Strategy 1: Pure follow
r1 = pattern_backtest(c_deep, lambda row: int(row['lag1']), "Pure Follow (v2)")

# Strategy 2: 3-lag pattern lookup
pat3_map = c_deep.groupby('pattern')['went_up'].mean().to_dict()
r2 = pattern_backtest(c_deep, lambda row: 1 if pat3_map.get(row['pattern'], 0.5) > 0.5 else 0,
                       "3-Lag Pattern")

# Strategy 3: 3-lag pattern + move size
combo_full_map = c_deep.groupby('combo')['went_up'].mean().to_dict()
r3 = pattern_backtest(c_deep, lambda row: 1 if combo_full_map.get(row['combo'], 0.5) > 0.5 else 0,
                       "3-Lag + Move Size")

results = [r1, r2, r3]
results.sort(key=lambda x: x['win_rate'], reverse=True)

print(f"  {'Strategy':<25} {'Win%':>6} {'Trades':>7} {'Busts':>6} {'$/hour':>8} {'Profit':>9}")
print(f"  {'-'*65}")
for r in results:
    marker = " ★" if r == results[0] else ""
    print(f"  {r['label']:<25} {r['win_rate']:>5.1f}% {r['trades']:>7} {r['busts']:>6} "
          f"${r['per_hour']:>7.2f} ${r['profit']:>8.2f}{marker}")

# Equity curves
fig, ax = plt.subplots(figsize=(16, 7))
for r in results:
    lw = 2 if r == results[0] else 1
    ax.plot(r['history'], label=f"{r['label']} ({r['win_rate']:.1f}%)", linewidth=lw)
ax.axhline(y=1000, color='yellow', linestyle='--', alpha=0.3)
ax.set_title('No-Skip Strategies — Martingale Equity Curves')
ax.set_xlabel('Trade #')
ax.set_ylabel('Balance ($)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 14. The Wall: Follow-Last is Already Optimal

**Proven:** For every possible lag pattern (UUU, UUD, UDU, etc.), following lag1 is ALWAYS the best direction. No adaptive follow/fade strategy can beat pure follow.

**The loss breakdown by move size:**
| Move Size | Win Rate | % of Trades | Contribution to Losses |
|-----------|----------|-------------|----------------------|
| Large     | ~99.5%   | ~33%        | Almost zero |
| Medium    | ~93.5%   | ~33%        | Small |
| Small     | ~67%     | ~33%        | **Almost all** |

**Conclusion:** To get past 87%, we need to either:
1. **Predict small moves before they happen** → convert them to skips (but user wants zero skips)
2. **Use additional data** we don't currently have (tick-level, cross-pair, time patterns)
3. **Improve accuracy on small moves** — is there ANY signal within small-move trades?

Let's explore what we can still try.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# IDEA 9: Can We Predict Small Moves BEFORE They Happen?
# ═══════════════════════════════════════════════════════════════
# If we know the next move will be small, we know we'll likely lose.
# Can we predict move size from what we can see at entry?

print("IDEA 9: Predicting Move Size — Can We See Small Moves Coming?")
print("=" * 60)

c_pred = df[df['second'] == 0][['close']].copy()
c_pred['future_close'] = c_pred['close'].shift(-12)
c_pred['went_up'] = (c_pred['future_close'] > c_pred['close']).astype(int)
c_pred['move_abs'] = (c_pred['future_close'] - c_pred['close']).abs()
c_pred['prev_move'] = c_pred['move_abs'].shift(1)
c_pred['prev_move_2'] = c_pred['move_abs'].shift(2)
c_pred['prev_move_3'] = c_pred['move_abs'].shift(3)

# Is the CURRENT move small? (what we want to predict)
c_pred = c_pred.dropna()
small_threshold = c_pred['move_abs'].quantile(0.33)  # Bottom third
c_pred['is_small'] = (c_pred['move_abs'] < small_threshold).astype(int)

print(f"Small move threshold: {small_threshold:.6f}")
print(f"Small moves: {c_pred['is_small'].mean():.1%} of all trades\n")

# Feature 1: Does last move size predict next move size?
print("--- Does previous move size predict next move size? ---\n")

c_pred['prev_was_small'] = (c_pred['prev_move'] < small_threshold).astype(int)
c_pred['prev_was_large'] = (c_pred['prev_move'] > c_pred['move_abs'].quantile(0.67)).astype(int)

for label, mask in [('After SMALL move', c_pred['prev_was_small'] == 1),
                     ('After MEDIUM move', (c_pred['prev_was_small'] == 0) & (c_pred['prev_was_large'] == 0)),
                     ('After LARGE move', c_pred['prev_was_large'] == 1)]:
    pct_small = c_pred.loc[mask, 'is_small'].mean()
    avg_move = c_pred.loc[mask, 'move_abs'].mean()
    n = mask.sum()
    print(f"  {label:<25}: {pct_small:.1%} chance next is small, avg move={avg_move:.6f} ({n} trades)")

# Autocorrelation of move sizes
from scipy.stats import pearsonr
for lag in [1, 2, 3, 5]:
    corr, pval = pearsonr(c_pred['move_abs'].values[lag:], c_pred['move_abs'].values[:-lag])
    sig = "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else ""))
    print(f"\n  Move size autocorrelation lag {lag}: r={corr:.3f} (p={pval:.4f}) {sig}")

# Feature 2: Time of day — are small moves more common at certain hours?
print("\n--- Small move probability by hour ---\n")
c_pred['hour'] = c_pred.index.hour
by_hour = c_pred.groupby('hour').agg(
    small_pct=('is_small', 'mean'),
    avg_move=('move_abs', 'mean'),
    count=('is_small', 'count'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(by_hour['hour'], by_hour['small_pct'], color='orange', alpha=0.7)
axes[0].axhline(y=c_pred['is_small'].mean(), color='yellow', linestyle='--', label=f"Average ({c_pred['is_small'].mean():.1%})")
axes[0].set_title('% of Small Moves by Hour')
axes[0].set_xlabel('Hour (UTC)')
axes[0].set_ylabel('% Small Moves')
axes[0].legend()

axes[1].bar(by_hour['hour'], by_hour['avg_move'] * 100000, color='cyan', alpha=0.7)  # In pips
axes[1].set_title('Average Move Size by Hour (pips)')
axes[1].set_xlabel('Hour (UTC)')
axes[1].set_ylabel('Average Move (pips × 10)')
plt.tight_layout()
plt.show()

# Feature 3: Does the SHAPE of the last move predict the SIZE of the next?
# Get last minute's intra-prices for volatility measure
print("\n--- Intra-minute volatility as predictor ---\n")

# Use existing pm_cols if available, otherwise build simple version
# Volatility of previous minute = std of 5s returns
c_pred['prev_volatility'] = c_pred['close'].rolling(12).std().shift(1)
c_pred['vol_q'] = pd.qcut(c_pred['prev_volatility'].dropna(), q=3, labels=['Low vol', 'Med vol', 'High vol'], duplicates='drop')

for label in ['Low vol', 'Med vol', 'High vol']:
    mask = c_pred['vol_q'] == label
    if mask.sum() > 50:
        pct_small = c_pred.loc[mask, 'is_small'].mean()
        follow_wr = (c_pred.loc[mask, 'went_up'] == c_pred.loc[mask, 'went_up'].shift(1)).mean()
        n = mask.sum()
        print(f"  {label:<12}: {pct_small:.1%} small, follow_wr={follow_wr:.1%} ({n} trades)")

# Feature 4: Consecutive small moves — do they cluster?
print("\n--- Do small moves cluster? ---\n")
small_streak = []
count = 0
for val in c_pred['is_small'].values:
    if val == 1:
        count += 1
    else:
        if count > 0:
            small_streak.append(count)
        count = 0
if count > 0:
    small_streak.append(count)

print(f"  Small move streaks: {len(small_streak)}")
print(f"  Average streak:     {np.mean(small_streak):.1f} minutes")
print(f"  Median streak:      {np.median(small_streak):.0f} minutes")
print(f"  Max streak:         {max(small_streak)} minutes")
print(f"  Streaks of 1:       {sum(1 for s in small_streak if s == 1)} ({sum(1 for s in small_streak if s == 1)/len(small_streak)*100:.0f}%)")
print(f"  Streaks of 3+:      {sum(1 for s in small_streak if s >= 3)} ({sum(1 for s in small_streak if s >= 3)/len(small_streak)*100:.0f}%)")

# KEY QUESTION: If we COULD perfectly predict small moves,
# and skip them, what would our win rate be?
non_small = c_pred[c_pred['is_small'] == 0]
follow_wr_non_small = (non_small['went_up'] == non_small['went_up'].shift(1)).mean()
print(f"\n  If we PERFECTLY predicted & skipped all small moves:")
print(f"    Win rate on remaining: {follow_wr_non_small:.1%}")
print(f"    Trades remaining:      {len(non_small)} ({len(non_small)/len(c_pred)*100:.0f}%)")
print(f"    Skip rate:             {c_pred['is_small'].mean()*100:.0f}%")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# IDEA 10: Within Small Moves — Is There ANY Signal?
# ═══════════════════════════════════════════════════════════════
# When the move is small, follow-last only works ~67%.
# Can we find a DIFFERENT signal for these trades?

print("IDEA 10: Finding Signal Within Small Moves")
print("=" * 60)

# Isolate small-move trades
small = c_pred[c_pred['is_small'] == 1].copy()
print(f"\nSmall-move trades: {len(small)}")
print(f"Follow-last accuracy on small: {(small['went_up'] == small['went_up'].shift(1)).dropna().mean():.1%}\n")

# What features exist for small-move trades?
small['prev_went_up'] = small['went_up'].shift(1)
small['prev_went_up_2'] = small['went_up'].shift(2)
small['prev_went_up_3'] = small['went_up'].shift(3)

# Momentum age within small moves
ages = []
count = 0
prev = None
for val in small['went_up'].values:
    if val == prev: count += 1
    else: count = 1
    ages.append(count)
    prev = val
small['age'] = ages
small['age'] = small['age'].shift(1)  # lag it

small = small.dropna(subset=['prev_went_up', 'age'])

# Test every direction signal for small-move trades specifically
print("--- Direction signals for SMALL MOVES only ---\n")

signals_small = []

# Follow lag1
acc = (small['went_up'] == small['prev_went_up']).mean()
signals_small.append(('Follow lag1', acc, len(small)))

# Fade lag1
acc = (small['went_up'] != small['prev_went_up']).mean()
signals_small.append(('Fade lag1', acc, len(small)))

# Follow lag2
valid = small.dropna(subset=['prev_went_up_2'])
acc = (valid['went_up'] == valid['prev_went_up_2']).mean()
signals_small.append(('Follow lag2', acc, len(valid)))

# Majority vote
valid = small.dropna(subset=['prev_went_up', 'prev_went_up_2', 'prev_went_up_3'])
majority = ((valid['prev_went_up'] + valid['prev_went_up_2'] + valid['prev_went_up_3']) >= 2).astype(int)
acc = (valid['went_up'] == majority).mean()
signals_small.append(('Majority of 3', acc, len(valid)))

# Always UP
acc = small['went_up'].mean()
signals_small.append(('Always UP', acc, len(small)))

# Always DOWN
acc = 1 - small['went_up'].mean()
signals_small.append(('Always DOWN', acc, len(small)))

# Follow based on age: if age >= 2, follow; if age == 1, fade
mask_old = small['age'] >= 2
mask_new = small['age'] == 1
if mask_old.sum() > 0 and mask_new.sum() > 0:
    correct_old = (small.loc[mask_old, 'went_up'] == small.loc[mask_old, 'prev_went_up']).sum()
    correct_new = (small.loc[mask_new, 'went_up'] != small.loc[mask_new, 'prev_went_up']).sum()
    total = len(small)
    acc = (correct_old + correct_new) / total
    signals_small.append(('Age≥2→follow, Age=1→fade', acc, total))

# By previous move size (even within "small" there's variation)
small['prev_move_q'] = pd.qcut(small['prev_move'], q=2, labels=['Tiny', 'Very Tiny'], duplicates='drop')
for label in ['Tiny', 'Very Tiny']:
    mask = small['prev_move_q'] == label
    if mask.sum() > 20:
        acc = (small.loc[mask, 'went_up'] == small.loc[mask, 'prev_went_up']).mean()
        signals_small.append((f'Follow (prev={label})', acc, mask.sum()))

signals_small.sort(key=lambda x: x[1], reverse=True)

print(f"  {'Signal':<30} {'Accuracy':>8} {'Trades':>7}")
print(f"  {'-'*50}")
for name, acc, n in signals_small:
    marker = " ★" if acc >= 0.70 else ""
    print(f"  {name:<30} {acc:>7.1%} {n:>7}{marker}")

# ═══════════════════════════════════════════════════════════════
# IDEA 11: Confidence-Weighted Staking (No Skips)
# ═══════════════════════════════════════════════════════════════
# Instead of skip vs trade, what about varying STAKE SIZE?
# High confidence (large prev move) → full stake
# Low confidence (small prev move) → minimum stake
# This effectively reduces losses from small moves without skipping

print(f"\n{'='*60}")
print("IDEA 11: Confidence-Weighted Staking (No Skip)")
print(f"{'='*60}\n")

c_stake = df[df['second'] == 0][['close']].copy()
c_stake['future_close'] = c_stake['close'].shift(-12)
c_stake['went_up'] = (c_stake['future_close'] > c_stake['close']).astype(int)
c_stake['prev_went_up'] = c_stake['went_up'].shift(1)
c_stake['move_abs'] = (c_stake['future_close'] - c_stake['close']).abs()
c_stake['prev_move'] = c_stake['move_abs'].shift(1)
c_stake = c_stake.dropna()

hours = len(df) * 5 / 3600

def confidence_staking_backtest(df_in, stake_fn, label=""):
    """Backtest with variable stake sizes based on confidence."""
    balance = 1000
    history = [balance]
    consec = 0
    wins = 0; losses = 0; busts = 0
    base_stake = 1.0
    martingale_stake = base_stake
    
    last_dir = None
    for _, row in df_in.iterrows():
        if last_dir is None:
            last_dir = int(row['went_up'])
            continue
        
        # Get confidence-adjusted base stake
        confidence_stake = stake_fn(row, base_stake)
        
        # Apply martingale on top of confidence stake
        if consec > 0:
            actual_stake = martingale_stake
        else:
            actual_stake = confidence_stake
            
        actual = int(row['went_up'])
        
        if consec >= 8 or actual_stake > 200:
            busts += 1; martingale_stake = base_stake; consec = 0
            actual_stake = confidence_stake
        if actual_stake > balance: break
        
        if last_dir == actual:
            balance += actual_stake * 0.85
            wins += 1; martingale_stake = base_stake; consec = 0
        else:
            balance -= actual_stake
            losses += 1; consec += 1
            martingale_stake = round((actual_stake + base_stake) / 0.85, 2)
        
        last_dir = actual
        history.append(balance)
    
    total = wins + losses
    profit = balance - 1000
    return {
        'label': label,
        'win_rate': wins / total * 100 if total > 0 else 0,
        'trades': total,
        'busts': busts,
        'profit': profit,
        'per_hour': profit / hours if hours > 0 else 0,
    }

# Different staking strategies
small_q = c_stake['prev_move'].quantile(0.33)
large_q = c_stake['prev_move'].quantile(0.67)

staking = []

# Baseline: flat $1
r = confidence_staking_backtest(c_stake, lambda row, bs: bs, "Flat $1")
staking.append(r)

# Small move → $0.50, else $1
r = confidence_staking_backtest(c_stake, 
    lambda row, bs: 0.50 if row['prev_move'] < small_q else bs,
    "Small→$0.50, else $1")
staking.append(r)

# Small → $0.25, Medium → $0.75, Large → $1.50
r = confidence_staking_backtest(c_stake, 
    lambda row, bs: 0.25 if row['prev_move'] < small_q else (1.50 if row['prev_move'] > large_q else 0.75),
    "Confidence tiers")
staking.append(r)

# Small → $0.10 (near-zero), else $1
r = confidence_staking_backtest(c_stake, 
    lambda row, bs: 0.10 if row['prev_move'] < small_q else bs,
    "Small→$0.10, else $1")
staking.append(r)

# Large → $2, else $1
r = confidence_staking_backtest(c_stake, 
    lambda row, bs: 2.0 if row['prev_move'] > large_q else bs,
    "Large→$2, else $1")
staking.append(r)

staking.sort(key=lambda x: x['per_hour'], reverse=True)

print(f"  {'Strategy':<25} {'Win%':>6} {'Trades':>7} {'Busts':>6} {'$/hour':>8} {'Profit':>9}")
print(f"  {'-'*65}")
for r in staking:
    marker = " ★" if r == staking[0] else ""
    print(f"  {r['label']:<25} {r['win_rate']:>5.1f}% {r['trades']:>7} {r['busts']:>6} "
          f"${r['per_hour']:>7.2f} ${r['profit']:>8.2f}{marker}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# IDEA 12: Aggressive Confidence Staking — Maximize $/hour
# ═══════════════════════════════════════════════════════════════
# Large moves have 99.5% win rate. We should bet WAY more on those.
# Small moves have 67% win rate. Minimize exposure there.

print("IDEA 12: Aggressive Confidence Staking Optimization")
print("=" * 60)

# First, confirm win rates by tier
c_tier = df[df['second'] == 0][['close']].copy()
c_tier['future_close'] = c_tier['close'].shift(-12)
c_tier['went_up'] = (c_tier['future_close'] > c_tier['close']).astype(int)
c_tier['prev_went_up'] = c_tier['went_up'].shift(1)
c_tier['move_abs'] = (c_tier['future_close'] - c_tier['close']).abs()
c_tier['prev_move'] = c_tier['move_abs'].shift(1)
c_tier = c_tier.dropna()

small_q = c_tier['prev_move'].quantile(0.33)
large_q = c_tier['prev_move'].quantile(0.67)

for label, mask in [('Small (<33%)', c_tier['prev_move'] < small_q),
                     ('Medium (33-67%)', (c_tier['prev_move'] >= small_q) & (c_tier['prev_move'] < large_q)),
                     ('Large (>67%)', c_tier['prev_move'] >= large_q)]:
    sub = c_tier[mask]
    wr = (sub['went_up'] == sub['prev_went_up']).mean()
    n = len(sub)
    # Expected value per $1 bet
    ev = wr * 0.85 - (1 - wr) * 1.0
    print(f"  {label:>20}: {wr:.1%} win rate, EV=${ev:.3f}/dollar, {n} trades")

hours = len(df) * 5 / 3600

print(f"\n--- Testing aggressive staking tiers ---\n")

staking_tests = []

# Sweep different multipliers for large moves
for large_mult in [1, 2, 3, 5, 7, 10]:
    for small_mult_pct in [100, 50, 25, 10]:
        small_stake = small_mult_pct / 100
        
        def make_fn(sm, lm, sq=small_q, lq=large_q):
            def fn(row, bs):
                if row['prev_move'] >= lq:
                    return bs * lm
                elif row['prev_move'] < sq:
                    return bs * sm
                else:
                    return bs
            return fn
        
        r = confidence_staking_backtest(c_tier, make_fn(small_stake, large_mult),
                                          f"S=${small_stake:.2f} L=${large_mult}")
        staking_tests.append(r)

staking_tests.sort(key=lambda x: x['per_hour'], reverse=True)

print(f"  {'Strategy':<20} {'Win%':>6} {'Trades':>7} {'Busts':>6} {'$/hour':>8} {'Profit':>10}")
print(f"  {'-'*60}")
for r in staking_tests[:15]:
    marker = " ★" if r == staking_tests[0] else ""
    print(f"  {r['label']:<20} {r['win_rate']:>5.1f}% {r['trades']:>7} {r['busts']:>6} "
          f"${r['per_hour']:>7.2f} ${r['profit']:>9.2f}{marker}")

# Also show the worst to understand the risk
print(f"\n  ... Bottom 3:")
for r in staking_tests[-3:]:
    print(f"  {r['label']:<20} {r['win_rate']:>5.1f}% {r['trades']:>7} {r['busts']:>6} "
          f"${r['per_hour']:>7.2f} ${r['profit']:>9.2f}")

best = staking_tests[0]
flat = [r for r in staking_tests if r['label'] == 'S=$1.00 L=$1'][0]
print(f"\n★ BEST: {best['label']}")
print(f"  Profit/hour: ${best['per_hour']:.2f} (vs flat ${flat['per_hour']:.2f})")
print(f"  Improvement: {best['per_hour']/flat['per_hour']*100 - 100:.0f}% more profit")
print(f"  Win rate:    {best['win_rate']:.1f}% (unchanged)")
print(f"  Busts:       {best['busts']}")

# Visualize: profit/hour heatmap by small_stake × large_mult
import itertools
small_vals = [0.10, 0.25, 0.50, 1.00]
large_vals = [1, 2, 3, 5, 7, 10]
heatmap = np.zeros((len(small_vals), len(large_vals)))
for i, sm in enumerate(small_vals):
    for j, lm in enumerate(large_vals):
        match = [r for r in staking_tests if r['label'] == f'S=${sm:.2f} L=${lm}']
        if match:
            heatmap[i, j] = match[0]['per_hour']

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(heatmap, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(large_vals)))
ax.set_xticklabels([f'${x}' for x in large_vals])
ax.set_yticks(range(len(small_vals)))
ax.set_yticklabels([f'${x:.2f}' for x in small_vals])
ax.set_xlabel('Large Move Stake')
ax.set_ylabel('Small Move Stake')
ax.set_title('Profit/Hour by Staking Configuration')

# Add values
for i in range(len(small_vals)):
    for j in range(len(large_vals)):
        ax.text(j, i, f'${heatmap[i,j]:.0f}', ha='center', va='center', fontsize=9,
                color='black' if heatmap[i,j] > heatmap.mean() else 'white')

plt.colorbar(im, label='$/hour')
plt.tight_layout()
plt.show()

## 15. Chasing 94% — What Are We Missing?

We've proven "follow last" is optimal at 87% with 5s candle data. The gap to 94% must come from something else entirely. Let's explore:

1. **Time windows** — are there hours/days where accuracy is naturally 94%+?
2. **Different result definition** — what if "direction" isn't :00→:00 but something else?
3. **Sub-candle signal** — is there information in the OHLC shape we're ignoring?
4. **Entry timing precision** — does entering at :01 vs :04 matter?

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TEST 1: Are there time windows where follow-last hits 94%?
# ═══════════════════════════════════════════════════════════════

print("TEST 1: Follow-Last Accuracy by Time Period")
print("=" * 60)

c = df[df['second'] == 0][['close']].copy()
c['future_close'] = c['close'].shift(-12)
c['went_up'] = (c['future_close'] > c['close']).astype(int)
c['prev_went_up'] = c['went_up'].shift(1)
c['follow_correct'] = (c['went_up'] == c['prev_went_up']).astype(int)
c = c.dropna()

# By hour
print("\n--- By Hour ---")
c['hour'] = c.index.hour
by_hour = c.groupby('hour').agg(
    win_rate=('follow_correct', 'mean'),
    count=('follow_correct', 'count'),
).reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['lime' if w >= 0.90 else ('gold' if w >= 0.87 else 'red') for w in by_hour['win_rate']]
ax.bar(by_hour['hour'], by_hour['win_rate'], color=colors, alpha=0.7)
ax.axhline(y=0.87, color='yellow', linestyle='--', label='Baseline (87%)')
ax.axhline(y=0.94, color='red', linestyle='--', label='Target (94%)')
ax.set_title('Follow-Last Win Rate by Hour')
ax.set_xlabel('Hour (UTC)')
ax.set_ylabel('Win Rate')
ax.legend()
ax.set_ylim(0.75, 1.0)
plt.tight_layout()
plt.show()

for _, row in by_hour.iterrows():
    marker = " ★" if row['win_rate'] >= 0.90 else ""
    print(f"  Hour {int(row['hour']):>2}: {row['win_rate']:.1%} ({int(row['count'])} trades){marker}")

# By day of week
print("\n--- By Day of Week ---")
c['dow'] = c.index.dayofweek
by_dow = c.groupby('dow').agg(
    win_rate=('follow_correct', 'mean'),
    count=('follow_correct', 'count'),
).reset_index()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
for _, row in by_dow.iterrows():
    marker = " ★" if row['win_rate'] >= 0.90 else ""
    print(f"  {days[int(row['dow'])]}: {row['win_rate']:.1%} ({int(row['count'])} trades){marker}")

# By date — any specific days hit 94%?
print("\n--- By Date (looking for 94%+ days) ---")
c['date'] = c.index.date
by_date = c.groupby('date').agg(
    win_rate=('follow_correct', 'mean'),
    count=('follow_correct', 'count'),
).reset_index()

for _, row in by_date.iterrows():
    marker = " ★★★" if row['win_rate'] >= 0.94 else (" ★" if row['win_rate'] >= 0.90 else "")
    print(f"  {row['date']}: {row['win_rate']:.1%} ({int(row['count'])} trades){marker}")

# Rolling window — does accuracy fluctuate?
print("\n--- Rolling 100-trade win rate ---")
c['rolling_wr'] = c['follow_correct'].rolling(100).mean()
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(c['rolling_wr'].values, color='cyan', linewidth=0.5)
ax.axhline(y=0.87, color='yellow', linestyle='--', label='87% baseline')
ax.axhline(y=0.94, color='red', linestyle='--', label='94% target')
ax.axhline(y=0.80, color='orange', linestyle='--', alpha=0.5)
ax.set_title('Rolling 100-Trade Win Rate Over Time')
ax.set_xlabel('Trade #')
ax.set_ylabel('Win Rate')
ax.legend()
ax.set_ylim(0.6, 1.0)
plt.tight_layout()
plt.show()

pct_above_90 = (c['rolling_wr'] >= 0.90).mean() * 100
pct_above_94 = (c['rolling_wr'] >= 0.94).mean() * 100
print(f"  % of time above 90%: {pct_above_90:.1f}%")
print(f"  % of time above 94%: {pct_above_94:.1f}%")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TEST 2: Different Result Definitions
# ═══════════════════════════════════════════════════════════════
# What if "direction" isn't measured :00→:00?
# IQ Option turbo options expire at specific timestamps.
# Maybe the result is measured differently.

print("TEST 2: Different Ways to Define 'Direction'")
print("=" * 60)

# We've been using: entry at :00, result = close at next :00
# But what if:

definitions = []

# Def 1: Our standard — :00 close to next :00 close (60s window)
c00 = df[df['second'] == 0][['close']].copy()
c00['result'] = c00['close'].shift(-12)
c00['went_up'] = (c00['result'] > c00['close']).astype(int)
c00['prev'] = c00['went_up'].shift(1)
c00_valid = c00.dropna()
acc = (c00_valid['went_up'] == c00_valid['prev']).mean()
definitions.append((':00→:00 (our standard)', acc, len(c00_valid)))

# Def 2: Entry at :00, result at :55 (55s window — avoid :00 boundary)
c00['result_55'] = None
c55 = df[df['second'] == 55][['close']].rename(columns={'close': 'close_55'})
for idx in c00.index:
    target = idx + pd.Timedelta(seconds=55)
    matches = c55.index.get_indexer([target], method='nearest')
    if matches[0] >= 0 and abs((c55.index[matches[0]] - target).total_seconds()) < 3:
        c00.loc[idx, 'result_55'] = c55.iloc[matches[0]]['close_55']
c00['result_55'] = pd.to_numeric(c00['result_55'], errors='coerce')
c00['went_up_55'] = (c00['result_55'] > c00['close']).astype(int)
c00['prev_55'] = c00['went_up_55'].shift(1)
valid = c00.dropna(subset=['went_up_55', 'prev_55'])
acc = (valid['went_up_55'] == valid['prev_55']).mean()
definitions.append((':00→:55 (55s window)', acc, len(valid)))

# Def 3: Entry at :00, result at :50 (50s window)
c00['result_50'] = None
c50 = df[df['second'] == 50][['close']].rename(columns={'close': 'close_50'})
for idx in c00.index:
    target = idx + pd.Timedelta(seconds=50)
    matches = c50.index.get_indexer([target], method='nearest')
    if matches[0] >= 0 and abs((c50.index[matches[0]] - target).total_seconds()) < 3:
        c00.loc[idx, 'result_50'] = c50.iloc[matches[0]]['close_50']
c00['result_50'] = pd.to_numeric(c00['result_50'], errors='coerce')
c00['went_up_50'] = (c00['result_50'] > c00['close']).astype(int)
c00['prev_50'] = c00['went_up_50'].shift(1)
valid = c00.dropna(subset=['went_up_50', 'prev_50'])
acc = (valid['went_up_50'] == valid['prev_50']).mean()
definitions.append((':00→:50 (50s window)', acc, len(valid)))

# Def 4: Entry at :05, result at next :05 (shifted window)
c05 = df[df['second'] == 5][['close']].copy()
c05['result'] = c05['close'].shift(-12)
c05['went_up'] = (c05['result'] > c05['close']).astype(int)
c05['prev'] = c05['went_up'].shift(1)
valid = c05.dropna()
acc = (valid['went_up'] == valid['prev']).mean()
definitions.append((':05→:05 (shifted 5s)', acc, len(valid)))

# Def 5: Use HIGH instead of CLOSE for direction
c00_hl = df[df['second'] == 0][['close', 'high', 'low']].copy()
c00_hl['future_high'] = c00_hl['high'].shift(-12)
c00_hl['future_low'] = c00_hl['low'].shift(-12)
# Did price go higher than entry at any point? (any-touch UP)
c00_hl['touched_up'] = (c00_hl['future_high'] > c00_hl['close']).astype(int)
c00_hl['prev_touched'] = c00_hl['touched_up'].shift(1)
valid = c00_hl.dropna()
acc = (valid['touched_up'] == valid['prev_touched']).mean()
definitions.append((':00 any-touch-UP (high)', acc, len(valid)))

# Def 6: Entry at :02 or :03 (typical human click delay)
for entry_sec in [0, 5, 10]:
    ahead = (60 - entry_sec) // 5
    c_e = df[df['second'] == entry_sec][['close']].copy()
    c_e['result'] = c_e['close'].shift(-ahead)
    c_e['went_up'] = (c_e['result'] > c_e['close']).astype(int)
    c_e['prev'] = c_e['went_up'].shift(1)
    valid = c_e.dropna()
    # But use the PREVIOUS :00→:00 result to decide direction
    # (this simulates seeing result at :00 but clicking at :02/:03)
    acc = (valid['went_up'] == valid['prev']).mean()
    definitions.append((f'Entry :{entry_sec:02d}, self-follow', acc, len(valid)))

# Def 7: Cross-minute — use :00→:00 result to predict :05→:05
c_cross = df[df['second'] == 5][['close']].copy()
c_cross['result'] = c_cross['close'].shift(-12)  # :05 to next :05
c_cross['went_up'] = (c_cross['result'] > c_cross['close']).astype(int)
# Get the :00→:00 result (which is "known" at :05)
c00_result = df[df['second'] == 0][['close']].copy()
c00_result['r00'] = c00_result['close'].shift(-12)
c00_result['dir_00'] = (c00_result['r00'] > c00_result['close']).astype(int)
# Align: for each :05, find the :00 result that just completed
for idx in c_cross.index:
    target = idx - pd.Timedelta(seconds=5)  # The :00 that just happened
    matches = c00_result.index.get_indexer([target], method='nearest')
    if matches[0] >= 0 and abs((c00_result.index[matches[0]] - target).total_seconds()) < 3:
        c_cross.loc[idx, 'prev_00_dir'] = c00_result.iloc[matches[0]]['dir_00']
c_cross['prev_00_dir'] = pd.to_numeric(c_cross['prev_00_dir'], errors='coerce')
valid = c_cross.dropna()
acc = (valid['went_up'] == valid['prev_00_dir']).mean()
definitions.append(('Use :00 result → predict :05→:05', acc, len(valid)))

definitions.sort(key=lambda x: x[1], reverse=True)

print(f"\n  {'Definition':<35} {'Accuracy':>8} {'Trades':>7}")
print(f"  {'-'*55}")
for name, acc, n in definitions:
    marker = " ★" if acc > 0.87 else ""
    print(f"  {name:<35} {acc:>7.1%} {n:>7}{marker}")

# Visualize
fig, ax = plt.subplots(figsize=(14, 6))
names = [d[0][:30] for d in definitions]
accs = [d[1] for d in definitions]
colors = ['lime' if a > 0.87 else ('gold' if a > 0.80 else 'cyan') for a in accs]
ax.barh(range(len(definitions)), accs, color=colors, alpha=0.7)
ax.axvline(x=0.87, color='yellow', linestyle='--', label='87% baseline')
ax.axvline(x=0.94, color='red', linestyle='--', label='94% target')
ax.set_yticks(range(len(definitions)))
ax.set_yticklabels(names, fontsize=8)
ax.set_title('Follow-Last Accuracy by Result Definition')
ax.set_xlabel('Accuracy')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# TEST 3: OHLC Shape Signal — Using More Than Just Close
# ═══════════════════════════════════════════════════════════════
# Each 5-second candle has Open, High, Low, Close.
# The relationship between these tells us about price action:
# - Long upper wick = sellers pushed back
# - Close = High = strong bullish pressure  
# - Close near Open = indecision

print("TEST 3: OHLC Shape Features at Entry")
print("=" * 60)

c_ohlc = df[df['second'] == 0][['open', 'high', 'low', 'close']].copy()
c_ohlc['future_close'] = c_ohlc['close'].shift(-12)
c_ohlc['went_up'] = (c_ohlc['future_close'] > c_ohlc['close']).astype(int)
c_ohlc['prev_went_up'] = c_ohlc['went_up'].shift(1)
c_ohlc['follow_correct'] = (c_ohlc['went_up'] == c_ohlc['prev_went_up']).astype(int)

# Features from the :00 candle itself (what we see RIGHT at entry)
candle_range = c_ohlc['high'] - c_ohlc['low']
c_ohlc['body'] = (c_ohlc['close'] - c_ohlc['open']).abs()
c_ohlc['body_pct'] = np.where(candle_range > 0, c_ohlc['body'] / candle_range, 0)
c_ohlc['upper_wick'] = c_ohlc['high'] - c_ohlc[['open', 'close']].max(axis=1)
c_ohlc['lower_wick'] = c_ohlc[['open', 'close']].min(axis=1) - c_ohlc['low']
c_ohlc['candle_bullish'] = (c_ohlc['close'] > c_ohlc['open']).astype(int)

# Previous candle (at :55) features
c55_ohlc = df[df['second'] == 55][['open', 'high', 'low', 'close']].copy()
for idx in c_ohlc.index:
    target = idx - pd.Timedelta(seconds=5)
    matches = c55_ohlc.index.get_indexer([target], method='nearest')
    if matches[0] >= 0 and abs((c55_ohlc.index[matches[0]] - target).total_seconds()) < 3:
        c_ohlc.loc[idx, 'prev_candle_bullish'] = int(c55_ohlc.iloc[matches[0]]['close'] > c55_ohlc.iloc[matches[0]]['open'])
        c_ohlc.loc[idx, 'prev_candle_body'] = abs(c55_ohlc.iloc[matches[0]]['close'] - c55_ohlc.iloc[matches[0]]['open'])

c_ohlc = c_ohlc.dropna()

# Does the :00 candle's shape predict anything?
print("\n--- :00 candle shape vs follow accuracy ---\n")

# Bullish vs bearish entry candle
for label, mask in [('Entry candle BULLISH', c_ohlc['candle_bullish'] == 1),
                     ('Entry candle BEARISH', c_ohlc['candle_bullish'] == 0)]:
    wr = c_ohlc.loc[mask, 'follow_correct'].mean()
    n = mask.sum()
    print(f"  {label:<30}: {wr:.1%} ({n} trades)")

# Does entry candle direction MATCHING momentum help?
c_ohlc['candle_matches_momentum'] = (
    ((c_ohlc['candle_bullish'] == 1) & (c_ohlc['prev_went_up'] == 1)) |
    ((c_ohlc['candle_bullish'] == 0) & (c_ohlc['prev_went_up'] == 0))
).astype(int)

print()
for label, mask in [('Entry candle MATCHES momentum', c_ohlc['candle_matches_momentum'] == 1),
                     ('Entry candle OPPOSES momentum', c_ohlc['candle_matches_momentum'] == 0)]:
    wr = c_ohlc.loc[mask, 'follow_correct'].mean()
    n = mask.sum()
    print(f"  {label:<35}: {wr:.1%} ({n} trades)")

# Body size of entry candle
print()
c_ohlc['body_q'] = pd.qcut(c_ohlc['body'], q=4, labels=['Doji', 'Small', 'Medium', 'Large'], duplicates='drop')
by_body = c_ohlc.groupby('body_q').agg(
    follow_wr=('follow_correct', 'mean'),
    count=('follow_correct', 'count'),
).reset_index()
for _, row in by_body.iterrows():
    marker = " ★" if row['follow_wr'] > 0.88 else ""
    print(f"  Entry body {row['body_q']:<8}: {row['follow_wr']:.1%} ({int(row['count'])} trades){marker}")

# ═══════════════════════════════════════════════════════════════
# TEST 4: Use Close-to-Open Gap as Signal
# ═══════════════════════════════════════════════════════════════
# The :00 candle's open vs close of the :55 candle
# If there's a gap, it might indicate momentum

print(f"\n{'='*60}")
print("TEST 4: Close→Open Gap at Minute Boundary")
print(f"{'='*60}\n")

# The gap between :55 close and :00 open
c_ohlc['gap'] = c_ohlc['open'] - c_ohlc['close'].shift(1)  # Approximate
c_ohlc['gap_up'] = (c_ohlc['gap'] > 0).astype(int)

# Does gap direction predict next minute?
gap_matches = (c_ohlc['went_up'] == c_ohlc['gap_up']).mean()
print(f"  Gap direction predicts next minute: {gap_matches:.1%}")
print(f"  (If >87%, this is useful)")

# Does gap matching momentum direction help?
c_ohlc['gap_matches_momentum'] = (c_ohlc['gap_up'] == c_ohlc['prev_went_up']).astype(int)

for label, mask in [('Gap CONFIRMS momentum', c_ohlc['gap_matches_momentum'] == 1),
                     ('Gap OPPOSES momentum', c_ohlc['gap_matches_momentum'] == 0)]:
    wr = c_ohlc.loc[mask, 'follow_correct'].mean()
    n = mask.sum()
    print(f"  {label:<30}: {wr:.1%} ({n} trades)")

# ═══════════════════════════════════════════════════════════════
# SUMMARY: All signals above 87%
# ═══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("ALL SIGNALS TESTED — Anything above 87%?")
print(f"{'='*60}\n")

all_signals = [
    ('Follow last result (baseline)', (c_ohlc['went_up'] == c_ohlc['prev_went_up']).mean()),
    ('Entry candle matches momentum', c_ohlc.loc[c_ohlc['candle_matches_momentum'] == 1, 'follow_correct'].mean()),
    ('Entry candle opposes momentum', c_ohlc.loc[c_ohlc['candle_matches_momentum'] == 0, 'follow_correct'].mean()),
    ('Gap confirms momentum', c_ohlc.loc[c_ohlc['gap_matches_momentum'] == 1, 'follow_correct'].mean()),
    ('Gap opposes momentum', c_ohlc.loc[c_ohlc['gap_matches_momentum'] == 0, 'follow_correct'].mean()),
]

all_signals.sort(key=lambda x: x[1], reverse=True)
for name, acc in all_signals:
    marker = " ★" if acc > 0.87 else ""
    print(f"  {name:<40}: {acc:.1%}{marker}")

## 16. The 96.1% Signal — Validate & Exploit

"Use :00 result → predict :05→:05" hit **96.1%**. But the :00→:00 and :05→:05 windows overlap by 55 seconds. Let's validate with properly separated windows and test if we can actually trade this.

**The idea:** Instead of entering at :00 and betting blind, wait until :05. By then you KNOW the confirmed :00 close price and the actual direction of the previous minute. Enter a 55-second trade at :05 using that fresh signal.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# VALIDATE: Fix the overlapping window issue
# ═══════════════════════════════════════════════════════════════
# The 96.1% used shift(-12) on filtered series which gives 12-minute
# windows, not 1-minute windows. Let's test properly.

print("VALIDATING: Proper 1-Minute Window Tests")
print("=" * 60)

# Build :00 candle series
c00 = df[df['second'] == 0][['close']].copy()
c05 = df[df['second'] == 5][['close']].rename(columns={'close': 'close_05'})
c55 = df[df['second'] == 55][['close']].rename(columns={'close': 'close_55'})

# For each :00, get the NEXT :00 close (1 minute later = shift(-1) on filtered series)
c00['next_00_close'] = c00['close'].shift(-1)
c00['went_up'] = (c00['next_00_close'] > c00['close']).astype(int)
c00['prev_went_up'] = c00['went_up'].shift(1)

# Also get the :05 and :55 prices
for idx in c00.index:
    # :05 of this minute (5 seconds after entry)
    t05 = idx + pd.Timedelta(seconds=5)
    m05 = c05.index.get_indexer([t05], method='nearest')
    if m05[0] >= 0 and abs((c05.index[m05[0]] - t05).total_seconds()) < 3:
        c00.loc[idx, 'this_05'] = c05.iloc[m05[0]]['close_05']
    
    # :55 of this minute (result at 55s)
    t55 = idx + pd.Timedelta(seconds=55)
    m55 = c55.index.get_indexer([t55], method='nearest')
    if m55[0] >= 0 and abs((c55.index[m55[0]] - t55).total_seconds()) < 3:
        c00.loc[idx, 'this_55'] = c55.iloc[m55[0]]['close_55']

c00['this_05'] = pd.to_numeric(c00['this_05'], errors='coerce')
c00['this_55'] = pd.to_numeric(c00['this_55'], errors='coerce')
c00 = c00.dropna()

print(f"Dataset: {len(c00):,} minutes with all prices\n")

# ── Test A: Standard follow-last (1-minute window) ──
acc_standard = (c00['went_up'] == c00['prev_went_up']).mean()
print(f"  A) Standard: :00→next:00, follow prev result")
print(f"     Accuracy: {acc_standard:.1%}")

# ── Test B: Enter at :05, predict :05→next:05 ──
# At :05 we KNOW the just-completed :00→:00 result
# Use that to predict :05→next:05
c05_full = df[df['second'] == 5][['close']].copy()
c05_full['next_05'] = c05_full['close'].shift(-1)  # Next :05 = 1 minute later
c05_full['went_up_05'] = (c05_full['next_05'] > c05_full['close']).astype(int)

# Get the :00→:00 result that JUST completed (known at :05)
for idx in c05_full.index:
    t00 = idx - pd.Timedelta(seconds=5)  # The :00 that just happened
    m00 = c00.index.get_indexer([t00], method='nearest')
    if m00[0] >= 0 and abs((c00.index[m00[0]] - t00).total_seconds()) < 3:
        c05_full.loc[idx, 'prev_00_dir'] = c00.iloc[m00[0]]['went_up']

c05_full['prev_00_dir'] = pd.to_numeric(c05_full['prev_00_dir'], errors='coerce')
c05_valid = c05_full.dropna()

acc_05 = (c05_valid['went_up_05'] == c05_valid['prev_00_dir']).mean()
print(f"\n  B) Enter :05, predict :05→next:05 using :00→:00 result")
print(f"     Accuracy: {acc_05:.1%}")

# ── Test C: Enter at :05, predict :05→next:00 (55s trade) ──
# This is what we'd actually trade: enter at :05, option expires at next :00
for idx in c05_full.index:
    t_next_00 = idx + pd.Timedelta(seconds=55)  # Next :00
    m = c00.index.get_indexer([t_next_00], method='nearest')
    if m[0] >= 0 and abs((c00.index[m[0]] - t_next_00).total_seconds()) < 3:
        c05_full.loc[idx, 'next_00_close'] = c00.iloc[m[0]]['close']

c05_full['next_00_close'] = pd.to_numeric(c05_full['next_00_close'], errors='coerce')
c05_full['went_up_to_00'] = (c05_full['next_00_close'] > c05_full['close']).astype(int)
c05_v2 = c05_full.dropna(subset=['went_up_to_00', 'prev_00_dir'])

acc_05_to_00 = (c05_v2['went_up_to_00'] == c05_v2['prev_00_dir']).mean()
print(f"\n  C) Enter :05, predict :05→next:00 (55s option) using :00→:00 result")
print(f"     Accuracy: {acc_05_to_00:.1%}")

# ── Test D: Self-follow at :05 (use own :05→:05 result) ──
c05_full['prev_05_dir'] = c05_full['went_up_05'].shift(1)
c05_self = c05_full.dropna(subset=['went_up_05', 'prev_05_dir'])
acc_05_self = (c05_self['went_up_05'] == c05_self['prev_05_dir']).mean()
print(f"\n  D) Self-follow: :05→next:05, follow own prev result")
print(f"     Accuracy: {acc_05_self:.1%}")

# ── Test E: Enter at :00, follow previous :00→:00 (use shift(-1)) ──
# This should be the TRUE 1-minute follow-last accuracy
acc_true = (c00['went_up'] == c00['prev_went_up']).mean()
print(f"\n  E) True 1-min: :00→next:00, follow prev :00→next:00 result")
print(f"     Accuracy: {acc_true:.1%}")

# ── Summary ──
print(f"\n{'='*60}")
print(f"SUMMARY — Proper 1-Minute Window Tests")
print(f"{'='*60}")
results = [
    ('A: :00→:00, follow prev', acc_standard),
    ('B: :05→:05, use :00 result', acc_05),
    ('C: :05→:00 (55s), use :00 result', acc_05_to_00),
    ('D: :05→:05, follow own prev', acc_05_self),
    ('E: True 1-min follow', acc_true),
]
results.sort(key=lambda x: x[1], reverse=True)
for name, acc in results:
    marker = " ★" if acc > 0.87 else ""
    print(f"  {name:<40}: {acc:.1%}{marker}")

print(f"\n  Previous '96.1%' was using 12-minute overlapping windows.")
print(f"  These are proper 1-minute non-overlapping windows.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CRITICAL: Understanding the Window Sizes
# ═══════════════════════════════════════════════════════════════
# Our original backtests used shift(-12) on minute-filtered data.
# On a series with 1 row per minute, shift(-12) = 12 MINUTES ahead!
# The correct shift for 1-minute predictions is shift(-1).
#
# Let's understand what's really going on:

print("CRITICAL: Follow-Last Accuracy by Prediction Window")
print("=" * 60)

c = df[df['second'] == 0][['close']].copy()

# Test follow-last at different window sizes (shift on filtered series)
window_results = []
for shift_val in [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20]:
    c[f'future_{shift_val}'] = c['close'].shift(-shift_val)
    c[f'up_{shift_val}'] = (c[f'future_{shift_val}'] > c['close']).astype(int)
    c[f'prev_up_{shift_val}'] = c[f'up_{shift_val}'].shift(1)
    valid = c.dropna(subset=[f'up_{shift_val}', f'prev_up_{shift_val}'])
    acc = (valid[f'up_{shift_val}'] == valid[f'prev_up_{shift_val}']).mean()
    window_results.append({
        'window_minutes': shift_val,
        'accuracy': acc,
        'count': len(valid),
    })

wr_df = pd.DataFrame(window_results)

fig, ax = plt.subplots(figsize=(14, 6))
colors = ['lime' if a > 0.54 else 'red' for a in wr_df['accuracy']]
ax.bar(wr_df['window_minutes'], wr_df['accuracy'], color=colors, alpha=0.7)
ax.axhline(y=0.5, color='white', linestyle=':', alpha=0.3, label='Random (50%)')
ax.axhline(y=0.5405, color='red', linestyle='--', label='Break-even (54%)')
ax.axhline(y=0.87, color='yellow', linestyle='--', label='Our "86.9%" (was 12-min window)')
ax.set_title('Follow-Last Accuracy by Prediction Window Size')
ax.set_xlabel('Window size (minutes)')
ax.set_ylabel('Follow-last accuracy')
ax.legend()
plt.tight_layout()
plt.show()

print(f"  {'Window':>8} {'Accuracy':>10} {'Profitable':>12}")
print(f"  {'-'*35}")
for _, row in wr_df.iterrows():
    profitable = "YES" if row['accuracy'] > 0.5405 else "no"
    marker = " ★ OUR ORIGINAL" if row['window_minutes'] == 12 else ""
    marker = " ← 1-MIN TRADE" if row['window_minutes'] == 1 else marker
    print(f"  {int(row['window_minutes']):>5} min {row['accuracy']:>9.1%} {profitable:>12}{marker}")

print(f"\n  KEY INSIGHT:")
print(f"  Our original backtests used shift(-12) = 12-minute windows.")
print(f"  IQ Option turbo options are 1-minute trades.")
print(f"  True 1-minute follow-last = {wr_df[wr_df['window_minutes']==1]['accuracy'].values[0]:.1%}")

# What's the MINIMUM window where follow-last is profitable?
min_profitable = wr_df[wr_df['accuracy'] > 0.5405]['window_minutes'].min()
print(f"  Minimum profitable window: {min_profitable} minutes")
print(f"  → Does IQ Option offer {min_profitable}-minute binary options?")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CROSS-CHECK: Is the shift bug real?
# ═══════════════════════════════════════════════════════════════
# Let's verify with raw data, no ambiguity.

print("CROSS-CHECK: Verifying the shift behavior")
print("=" * 60)

# Step 1: What does the filtered series actually look like?
c00 = df[df['second'] == 0][['close']].copy()
print(f"\nFiltered :00 series:")
print(f"  Total rows: {len(c00)}")
print(f"  First 5 timestamps:")
for i, idx in enumerate(c00.index[:5]):
    print(f"    Row {i}: {idx}  close={c00.iloc[i]['close']:.5f}")

# Step 2: What does shift(-12) actually give us?
print(f"\n  Row 0 close:            {c00.iloc[0]['close']:.5f} at {c00.index[0]}")
print(f"  Row 0 shift(-1) close:  {c00.iloc[1]['close']:.5f} at {c00.index[1]}")
print(f"  Row 0 shift(-12) close: {c00.iloc[12]['close']:.5f} at {c00.index[12]}")

time_diff_1 = (c00.index[1] - c00.index[0]).total_seconds()
time_diff_12 = (c00.index[12] - c00.index[0]).total_seconds()
print(f"\n  Time between row 0 and row 1:  {time_diff_1:.0f} seconds ({time_diff_1/60:.1f} minutes)")
print(f"  Time between row 0 and row 12: {time_diff_12:.0f} seconds ({time_diff_12/60:.1f} minutes)")

print(f"\n  → shift(-1) = {time_diff_1/60:.0f}-minute window")
print(f"  → shift(-12) = {time_diff_12/60:.0f}-minute window")
print(f"  → OUR BACKTEST USED shift(-12) = {time_diff_12/60:.0f}-MINUTE WINDOW")

# Step 3: Compare shift-based vs timestamp-based lookups
print(f"\n{'='*60}")
print("VERIFICATION: Three ways to measure follow-last accuracy")
print(f"{'='*60}")

# Method 1: shift(-12) on filtered series (OUR ORIGINAL)
c00['future_shift12'] = c00['close'].shift(-12)
c00['up_shift12'] = (c00['future_shift12'] > c00['close']).astype(int)
c00['prev_shift12'] = c00['up_shift12'].shift(1)
valid = c00.dropna(subset=['up_shift12', 'prev_shift12'])
acc_shift12 = (valid['up_shift12'] == valid['prev_shift12']).mean()

# Method 2: shift(-1) on filtered series (TRUE 1-minute)
c00['future_shift1'] = c00['close'].shift(-1)
c00['up_shift1'] = (c00['future_shift1'] > c00['close']).astype(int)
c00['prev_shift1'] = c00['up_shift1'].shift(1)
valid = c00.dropna(subset=['up_shift1', 'prev_shift1'])
acc_shift1 = (valid['up_shift1'] == valid['prev_shift1']).mean()

# Method 3: Timestamp-based lookup — explicitly find price 60 seconds later
c00['future_ts'] = None
for i, idx in enumerate(c00.index):
    target = idx + pd.Timedelta(seconds=60)
    # Find the closest :00 candle to 60 seconds later
    matches = c00.index.get_indexer([target], method='nearest')
    if matches[0] >= 0:
        actual_time = c00.index[matches[0]]
        time_diff = abs((actual_time - target).total_seconds())
        if time_diff < 5:  # Must be within 5 seconds
            c00.iloc[i, c00.columns.get_loc('future_ts')] = c00.iloc[matches[0]]['close']

c00['future_ts'] = pd.to_numeric(c00['future_ts'], errors='coerce')
c00['up_ts'] = (c00['future_ts'] > c00['close']).astype(int)
c00['prev_ts'] = c00['up_ts'].shift(1)
valid = c00.dropna(subset=['up_ts', 'prev_ts'])
acc_ts = (valid['up_ts'] == valid['prev_ts']).mean()

# Method 4: Use the FULL 5-second dataframe — shift(-12) on UNFILTERED df
# At each :00, the price 60 seconds later is 12 rows ahead in the FULL df
c00_v2 = pd.DataFrame(index=c00.index)
c00_v2['close'] = c00['close']
for i, idx in enumerate(c00.index):
    # Find this timestamp in the full df
    full_idx = df.index.get_indexer([idx], method='nearest')[0]
    if full_idx >= 0 and full_idx + 12 < len(df):
        future_idx = full_idx + 12  # 12 five-second candles = 60 seconds
        c00_v2.iloc[i, c00_v2.columns.get_loc('close')] = c00.iloc[i]['close']
        actual_future_time = df.index[future_idx]
        time_check = (actual_future_time - idx).total_seconds()
        if abs(time_check - 60) < 5:
            c00_v2.loc[idx, 'future_full'] = df.iloc[future_idx]['close']

c00_v2['future_full'] = pd.to_numeric(c00_v2['future_full'], errors='coerce')
c00_v2['up_full'] = (c00_v2['future_full'] > c00_v2['close']).astype(int)
c00_v2['prev_full'] = c00_v2['up_full'].shift(1)
valid = c00_v2.dropna(subset=['up_full', 'prev_full'])
acc_full = (valid['up_full'] == valid['prev_full']).mean()

print(f"\n  Method 1: shift(-12) on filtered (OUR BACKTEST):    {acc_shift12:.1%}  ← {time_diff_12/60:.0f}-min window")
print(f"  Method 2: shift(-1) on filtered:                    {acc_shift1:.1%}  ← 1-min window")
print(f"  Method 3: Timestamp lookup (+60s):                  {acc_ts:.1%}  ← 1-min window")
print(f"  Method 4: shift(-12) on FULL 5s df:                 {acc_full:.1%}  ← 1-min window")

print(f"\n  VERDICT:")
if abs(acc_shift1 - acc_ts) < 0.02 and abs(acc_ts - acc_full) < 0.02:
    if acc_shift1 < 0.55:
        print(f"  ❌ Methods 2, 3, 4 all agree: true 1-minute follow-last ≈ {acc_shift1:.1%}")
        print(f"  ❌ Method 1 (our backtest) was measuring {time_diff_12/60:.0f}-minute windows")
        print(f"  ❌ The backtest bug IS REAL.")
    else:
        print(f"  ✓ 1-minute follow-last is profitable at {acc_shift1:.1%}")
else:
    print(f"  ⚠ Methods disagree — need more investigation")
    print(f"    shift(-1)={acc_shift1:.1%}, timestamp={acc_ts:.1%}, full_df={acc_full:.1%}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CHECK: What is the actual candle size in our data?
# ═══════════════════════════════════════════════════════════════

print("RAW DATA CHECK: What are the actual candle intervals?")
print("=" * 60)

# Look at the first 20 rows of the FULL dataframe
print("\nFirst 15 rows of raw data:")
print(f"  {'Index':<25} {'Second':>6} {'Close':>10}")
print(f"  {'-'*45}")
for i in range(min(15, len(df))):
    print(f"  {str(df.index[i]):<25} {df.iloc[i]['second']:>6} {df.iloc[i]['close']:>10.5f}")

# Time differences between consecutive rows
diffs = pd.Series(df.index).diff().dt.total_seconds().dropna()
print(f"\nTime between consecutive candles:")
print(f"  Most common interval: {diffs.mode().values[0]:.0f} seconds")
print(f"  Mean interval: {diffs.mean():.1f} seconds")
print(f"  Min: {diffs.min():.0f}s, Max: {diffs.max():.0f}s")

# Distribution
print(f"\n  Interval distribution:")
for interval, count in diffs.value_counts().head(5).items():
    print(f"    {interval:.0f}s: {count} occurrences ({count/len(diffs)*100:.1f}%)")

# So when we filter df[df['second'] == 0], what do we get?
c00 = df[df['second'] == 0]
c00_diffs = pd.Series(c00.index).diff().dt.total_seconds().dropna()
print(f"\nFiltered :00 series intervals:")
print(f"  Most common: {c00_diffs.mode().values[0]:.0f} seconds")
print(f"  Mean: {c00_diffs.mean():.1f} seconds")

print(f"\n  Total candles in full df: {len(df)}")
print(f"  Total :00 candles: {len(c00)}")
print(f"  Ratio: {len(df)/len(c00):.1f} (should be ~12 if 5s candles)")

# KEY QUESTION: Is shift(-12) on the FULL df = 60 seconds?
print(f"\nFull df: row 0 = {df.index[0]}, row 12 = {df.index[12]}")
print(f"  Difference: {(df.index[12] - df.index[0]).total_seconds():.0f} seconds")
print(f"\nFiltered :00: row 0 = {c00.index[0]}, row 12 = {c00.index[12]}")
print(f"  Difference: {(c00.index[12] - c00.index[0]).total_seconds():.0f} seconds")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CORRECTED BACKTEST: 1-min, 2-min, 5-min with martingale
# ═══════════════════════════════════════════════════════════════

print("CORRECTED BACKTEST — Follow Last Result with Martingale")
print("=" * 70)

hours = len(df) * 5 / 3600

def corrected_backtest(df, window_minutes, base_stake=1.0, payout=0.85, 
                        max_losses=8, max_exposure=200, label=""):
    """
    Corrected backtest using proper time windows.
    Uses the FULL 5-second df with shift(-12*window_minutes) for correct timing.
    """
    # Get entry candles at :00
    c = df[df['second'] == 0][['close']].copy()
    
    # Find the close price window_minutes later using the FULL df
    c['future_close'] = None
    for i, idx in enumerate(c.index):
        target = idx + pd.Timedelta(minutes=window_minutes)
        # Look up in the full df
        matches = df.index.get_indexer([target], method='nearest')
        if matches[0] >= 0:
            actual_time = df.index[matches[0]]
            if abs((actual_time - target).total_seconds()) < 5:
                c.iloc[i, c.columns.get_loc('future_close')] = df.iloc[matches[0]]['close']
    
    c['future_close'] = pd.to_numeric(c['future_close'], errors='coerce')
    c['went_up'] = (c['future_close'] > c['close']).astype(int)
    c['move_abs'] = (c['future_close'] - c['close']).abs()
    c = c.dropna()
    
    # Simulate trading
    balance = 1000
    history = [balance]
    stake = base_stake
    consec = 0
    wins = 0
    losses = 0
    busts = 0
    last_dir = None
    
    # Trade every window_minutes (not every minute)
    trade_indices = list(range(0, len(c), window_minutes))
    
    for i in trade_indices:
        if i >= len(c):
            break
        row = c.iloc[i]
        actual = int(row['went_up'])
        
        if last_dir is None:
            last_dir = actual
            continue
        
        bet = last_dir
        
        # Martingale safety
        if consec >= max_losses or stake > max_exposure:
            busts += 1
            stake = base_stake
            consec = 0
        
        if stake > balance:
            break
        
        if bet == actual:
            balance += stake * payout
            wins += 1
            stake = base_stake
            consec = 0
        else:
            balance -= stake
            losses += 1
            consec += 1
            stake = round((stake + base_stake) / payout, 2)
        
        last_dir = actual
        history.append(balance)
    
    total = wins + losses
    profit = balance - 1000
    trades_per_hour = total / hours if hours > 0 else 0
    
    return {
        'label': label,
        'window': window_minutes,
        'win_rate': wins / total * 100 if total > 0 else 0,
        'trades': total,
        'trades_per_hour': trades_per_hour,
        'busts': busts,
        'profit': profit,
        'per_hour': profit / hours if hours > 0 else 0,
        'max_dd': 1000 - min(history),
        'history': history,
        'wins': wins,
        'losses': losses,
    }

# Run for each window
results = []
for window in [1, 2, 3, 5, 10, 12, 15]:
    r = corrected_backtest(df, window, label=f"{window}-min")
    results.append(r)

print(f"\nData: {hours:.0f} hours | $1 stakes | 85% payout | 8-level martingale")
print(f"{'='*90}")
print(f"{'Window':<8} {'Win%':>6} {'Trades':>7} {'Tr/hr':>6} {'Busts':>6} {'MaxDD':>8} {'$/hour':>8} {'Profit':>10}")
print(f"{'-'*90}")
for r in results:
    marker = ""
    if r['win_rate'] < 54: marker = " ❌ UNPROFITABLE"
    elif r['busts'] == 0: marker = " ✓ NO BUSTS"
    elif r['busts'] <= 5: marker = f" ⚠ {r['busts']} busts"
    else: marker = f" ❌ {r['busts']} BUSTS"
    
    print(f"{r['label']:<8} {r['win_rate']:>5.1f}% {r['trades']:>7} {r['trades_per_hour']:>5.1f} "
          f"{r['busts']:>6} ${r['max_dd']:>7.2f} ${r['per_hour']:>7.2f} ${r['profit']:>9.2f}{marker}")

# Equity curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for r in results:
    if r['window'] in [1, 2, 3, 5]:
        lw = 2 if r['window'] == 5 else 1
        axes[0].plot(r['history'], label=f"{r['label']} ({r['win_rate']:.1f}%, {r['busts']} busts)", linewidth=lw)
axes[0].axhline(y=1000, color='yellow', linestyle='--', alpha=0.3)
axes[0].set_title('Corrected Backtests: 1-5 min')
axes[0].set_xlabel('Trade #')
axes[0].set_ylabel('Balance ($)')
axes[0].legend(fontsize=8)

for r in results:
    if r['window'] in [5, 10, 12, 15]:
        lw = 2 if r['window'] == 5 else 1
        axes[1].plot(r['history'], label=f"{r['label']} ({r['win_rate']:.1f}%, {r['busts']} busts)", linewidth=lw)
axes[1].axhline(y=1000, color='yellow', linestyle='--', alpha=0.3)
axes[1].set_title('Corrected Backtests: 5-15 min')
axes[1].set_xlabel('Trade #')
axes[1].set_ylabel('Balance ($)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Summary
print(f"\nSUMMARY:")
for r in results:
    if r['window'] in [1, 2, 5]:
        print(f"\n  {r['label']}:")
        print(f"    Win rate:      {r['win_rate']:.1f}%")
        print(f"    Trades:        {r['trades']} ({r['trades_per_hour']:.1f}/hr)")
        print(f"    Busts (8L):    {r['busts']}")
        print(f"    Max drawdown:  ${r['max_dd']:.2f}")
        print(f"    Profit/hour:   ${r['per_hour']:.2f}")
        print(f"    Total profit:  ${r['profit']:.2f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CROSS-CHECK THE CORRECTED BACKTEST
# ═══════════════════════════════════════════════════════════════
# Is the corrected backtest itself correct? Let's verify multiple ways.

print("CROSS-CHECKING THE CORRECTED BACKTEST")
print("=" * 60)

# ── Check 1: Simple non-overlapping accuracy (no backtest, just counting) ──
print("\nCheck 1: Simple accuracy count (no martingale, just direction)")
print("-" * 60)

c = df[df['second'] == 0][['close']].copy()

for window in [1, 2, 5, 12]:
    # Get close price N minutes later via timestamp
    future_prices = []
    for idx in c.index:
        target = idx + pd.Timedelta(minutes=window)
        m = df.index.get_indexer([target], method='nearest')
        if m[0] >= 0 and abs((df.index[m[0]] - target).total_seconds()) < 5:
            future_prices.append(df.iloc[m[0]]['close'])
        else:
            future_prices.append(None)
    
    c[f'fut_{window}'] = future_prices
    c[f'fut_{window}'] = pd.to_numeric(c[f'fut_{window}'], errors='coerce')
    c[f'up_{window}'] = (c[f'fut_{window}'] > c['close']).astype(int)
    
    valid = c.dropna(subset=[f'up_{window}'])
    
    # NON-OVERLAPPING: take every Nth row
    non_overlap = valid.iloc[::window]
    directions = non_overlap[f'up_{window}'].values
    # Follow-last accuracy: does direction[i] == direction[i-1]?
    matches = sum(1 for i in range(1, len(directions)) if directions[i] == directions[i-1])
    total = len(directions) - 1
    acc_non_overlap = matches / total if total > 0 else 0
    
    # OVERLAPPING: every row (our old method)
    directions_all = valid[f'up_{window}'].values
    matches_all = sum(1 for i in range(1, len(directions_all)) if directions_all[i] == directions_all[i-1])
    total_all = len(directions_all) - 1
    acc_overlap = matches_all / total_all if total_all > 0 else 0
    
    print(f"  {window:>2}-min:  Overlapping={acc_overlap:.1%} ({total_all} pairs)  "
          f"Non-overlapping={acc_non_overlap:.1%} ({total} pairs)  "
          f"Δ={acc_overlap - acc_non_overlap:+.1%}")

# ── Check 2: Verify with synthetic random walk ──
print(f"\nCheck 2: Compare with a RANDOM WALK (same analysis)")
print("-" * 60)
print("If our OTC data gives same results as random walk,")
print("there's no real pattern to exploit.\n")

np.random.seed(42)
# Generate random walk with same length and step size as OTC data
steps = np.random.choice([-1, 1], size=len(df)) * 0.00010
random_prices = 1.14000 + np.cumsum(steps)

for window in [1, 2, 5, 12]:
    # Sample every minute (every 12th point)
    minute_prices = random_prices[::12]
    
    # Direction over window
    directions = []
    for i in range(len(minute_prices) - window):
        directions.append(int(minute_prices[i + window] > minute_prices[i]))
    
    # Non-overlapping follow-last
    non_overlap_dirs = directions[::window]
    matches = sum(1 for i in range(1, len(non_overlap_dirs)) if non_overlap_dirs[i] == non_overlap_dirs[i-1])
    total = len(non_overlap_dirs) - 1
    acc_rw_non = matches / total if total > 0 else 0
    
    # Overlapping follow-last
    matches_ol = sum(1 for i in range(1, len(directions)) if directions[i] == directions[i-1])
    total_ol = len(directions) - 1
    acc_rw_ol = matches_ol / total_ol if total_ol > 0 else 0
    
    print(f"  {window:>2}-min:  Overlapping={acc_rw_ol:.1%}  Non-overlapping={acc_rw_non:.1%}")

# ── Check 3: Raw autocorrelation of 1-minute returns ──
print(f"\nCheck 3: Autocorrelation of 1-minute price changes")
print("-" * 60)

c_ret = df[df['second'] == 0][['close']].copy()
c_ret['next_close'] = c_ret['close'].shift(-1)  # 1 minute later
c_ret['return_1m'] = c_ret['next_close'] - c_ret['close']
c_ret = c_ret.dropna()

from scipy.stats import pearsonr
for lag in [1, 2, 3, 5, 10]:
    returns = c_ret['return_1m'].values
    corr, pval = pearsonr(returns[lag:], returns[:-lag])
    sig = "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else "ns"))
    print(f"  Lag {lag:>2}: r={corr:+.4f} (p={pval:.4f}) {sig}")

print(f"\n  If r ≈ 0 at all lags, consecutive minute returns are independent")
print(f"  = no momentum = follow-last can't work")

# ── Check 4: Does the OLD buggy result replicate? ──
print(f"\nCheck 4: Replicate the original 86.9% (buggy shift)")
print("-" * 60)
c_old = df[df['second'] == 0][['close']].copy()
c_old['future'] = c_old['close'].shift(-12)  # THE BUG: 12 minutes, not 60 seconds
c_old['up'] = (c_old['future'] > c_old['close']).astype(int)
c_old['prev'] = c_old['up'].shift(1)
valid = c_old.dropna()
acc_old = (valid['up'] == valid['prev']).mean()

# Why is this 87%? Because consecutive OVERLAPPING 12-min windows share 11 minutes
# For a random walk: P(same direction) ≈ correlation of overlapping sums
# Expected: roughly (N-1)/N where N=12 → 11/12 = 91.7%
expected_rw = 11/12
print(f"  Buggy shift(-12) result:           {acc_old:.1%}")
print(f"  Expected for random walk overlap:  {expected_rw:.1%}")
print(f"  Our data is {'close to' if abs(acc_old - expected_rw) < 0.05 else 'different from'} random walk expectation")

print(f"\n{'='*60}")
print("FINAL VERDICT")
print(f"{'='*60}")